In [1]:
#chunk 0
#1
# Import the libraries we need for this check
import torch      # the deep learning library (already installed in Colab)
import psutil     # a small tool that reports memory (already installed in Colab)

# Print the PyTorch version, just so we know what we are working with
print("PyTorch version:", torch.__version__)

# Check whether a GPU is attached. For this project we WANT this to say False,
# because we are saving GPU time for only when we absolutely need it.
print("GPU attached:", torch.cuda.is_available())

# Show total RAM (memory) in gigabytes so we know our limits
ram_gb = psutil.virtual_memory().total / 1e9
print(f"Total RAM: {ram_gb:.1f} GB")

# Show how much disk space is free on Colab's local disk
disk = psutil.disk_usage("/content")
print(f"Free local disk: {disk.free / 1e9:.1f} GB")

PyTorch version: 2.14.1+cpu
GPU attached: False
Total RAM: 13.6 GB
Free local disk: 115.6 GB


In [2]:
#chunk 0
#2
# This lets Colab read and write files in your Google Drive
from google.colab import drive

# A popup will ask permission. Accept it. Files saved under
# /content/drive/MyDrive/ stay safe even if Colab disconnects.
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
#chunk 0
#3
import os  # os lets us create and check folders

# The main project folder, stored in your Drive so it survives disconnects
PROJECT = "/content/drive/MyDrive/mfcad_project"

# The sub-folders we will use, and what each is for
folders = [
    "data",     # converted/cached dataset files (chunk 3 saves here)
    "models",   # saved model checkpoints (chunks 4, 5, 6)
    "results",  # metrics, plots, confusion matrices (chunk 7)
    "notes",    # small text/JSON notes about what we found
]

# Create the main folder and every sub-folder.
# exist_ok=True means "do nothing if it already exists", so re-running is safe.
for f in folders:
    os.makedirs(os.path.join(PROJECT, f), exist_ok=True)

# Print what we created so you can confirm it worked
print("Created:", PROJECT)
print("Contents:", os.listdir(PROJECT))

Created: /content/drive/MyDrive/mfcad_project
Contents: ['data', 'models', 'results', 'notes', 'helpers.py', '__pycache__', 'progress.json']


In [4]:
#chunk 0
#4
%%writefile /content/drive/MyDrive/mfcad_project/helpers.py
# helpers.py - small tools for saving work and tracking progress.
# This file lives in Drive, so it is still there after a disconnect.

import json   # json lets us save simple Python data (dicts, lists) as text
import os     # os lets us build file paths
import time   # time lets us stamp when something was saved

# Where everything is stored
PROJECT = "/content/drive/MyDrive/mfcad_project"

# The file that remembers which chunks are finished
PROGRESS_FILE = os.path.join(PROJECT, "progress.json")


def save_json(obj, subfolder, filename):
    """Save a dict/list as a .json file in a project sub-folder."""
    path = os.path.join(PROJECT, subfolder, filename)   # build the full path
    with open(path, "w") as f:                          # open the file for writing
        json.dump(obj, f, indent=2)                     # write it, nicely formatted
    print("Saved:", path)
    return path


def load_json(subfolder, filename):
    """Load a .json file saved earlier. Returns None if it does not exist."""
    path = os.path.join(PROJECT, subfolder, filename)
    if not os.path.exists(path):                        # file missing? say so
        return None
    with open(path) as f:                               # open for reading
        return json.load(f)                             # turn the text back into data


def _read_progress():
    """Read the progress file (an empty dict if it does not exist yet)."""
    if not os.path.exists(PROGRESS_FILE):
        return {}
    with open(PROGRESS_FILE) as f:
        return json.load(f)


def mark_done(chunk_name, note=""):
    """Record that a chunk is finished, with a time stamp and optional note."""
    progress = _read_progress()                         # load what we have so far
    progress[chunk_name] = {
        "finished_at": time.strftime("%Y-%m-%d %H:%M:%S"),
        "note": note,
    }
    with open(PROGRESS_FILE, "w") as f:                 # write it back to Drive
        json.dump(progress, f, indent=2)
    print(f"Marked done: {chunk_name}")


def is_done(chunk_name):
    """True if this chunk was already finished (so we can skip redoing it)."""
    return chunk_name in _read_progress()


def show_progress():
    """Print every finished chunk and when it finished."""
    progress = _read_progress()
    if not progress:
        print("No chunks finished yet.")
    for name, info in progress.items():
        print(f"{name}: finished {info['finished_at']}  {info['note']}")

Overwriting /content/drive/MyDrive/mfcad_project/helpers.py


In [5]:
#chunk 0
# 5
# Tell Python to also look in our project folder when importing
import sys
sys.path.append("/content/drive/MyDrive/mfcad_project")

# Import our helper tools from the file we just wrote
import helpers

# Test 1: save a tiny note to Drive
helpers.save_json({"hello": "world"}, "notes", "test_note.json")

# Test 2: load it back and confirm it matches
print("Loaded back:", helpers.load_json("notes", "test_note.json"))

# Mark this chunk as finished in the progress file
helpers.mark_done("chunk0_setup", note="folders and helpers created")

# Show the progress list (should show chunk0_setup)
helpers.show_progress()

Saved: /content/drive/MyDrive/mfcad_project/notes/test_note.json
Loaded back: {'hello': 'world'}
Marked done: chunk0_setup
chunk0_setup: finished 2026-11-23 11:07:56  folders and helpers created


In [7]:
#chunk 1
#1
# Mount Google Drive so Colab can see your files (accept the popup)
from google.colab import drive
drive.mount('/content/drive')

# Tell Python where our project folder is, then load our helper tools
import sys
sys.path.append("/content/drive/MyDrive/mfcad_project")
import helpers

# Show finished chunks (chunk0_setup should appear)
helpers.show_progress()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
chunk0_setup: finished 2026-11-23 11:07:56  folders and helpers created


In [8]:
#chunk 1
# 2
import os, zipfile

# Where the zip should be. If your filename is different, edit it here.
ZIP_PATH = "/content/drive/MyDrive/mfcad_project/data/MFCAD_dataset.zip"

if not os.path.exists(ZIP_PATH):
    # The file is not where we expected. List what IS in the data folder.
    print("Zip not found. Files in the data folder:")
    print(os.listdir("/content/drive/MyDrive/mfcad_project/data"))
else:
    # The real size should be about 1500 MB
    print(f"Size: {os.path.getsize(ZIP_PATH) / 1e6:.1f} MB (expected about 1500 MB)")
    # is_zipfile checks that the archive is complete and readable
    print("Valid zip:", zipfile.is_zipfile(ZIP_PATH))

Size: 1615.8 MB (expected about 1500 MB)
Valid zip: True


In [9]:
#chunk 1
# 3

# Unzip onto Colab's fast local disk. This takes a few minutes and may print
# nothing until it ends. This folder is erased when the session ends, but the
# zip stays safe in Drive, so we can simply unzip again next time.
# The marker file .unzipped_ok tells us the unzip fully finished.
!test -f /content/mfcad/.unzipped_ok && echo "Already unzipped, skipping." || (mkdir -p /content/mfcad && unzip -q -o "{ZIP_PATH}" -d /content/mfcad && touch /content/mfcad/.unzipped_ok && echo "Unzipped successfully.")

Unzipped successfully.


In [10]:
#chunk 1
#4
import os, collections

ROOT = "/content/mfcad"                    # where we unzipped the dataset

ext_count = collections.Counter()          # how many files of each type
dir_counts = collections.defaultdict(collections.Counter)  # types per folder
key_files = []                             # the .txt and .h5 files we care about

# Walk through every folder and file inside the dataset
for dirpath, dirnames, filenames in os.walk(ROOT):
    rel_dir = os.path.relpath(dirpath, ROOT)       # folder name relative to ROOT
    for fn in filenames:
        ext = os.path.splitext(fn)[1].lower()      # file type, e.g. ".step"
        ext_count[ext] += 1
        dir_counts[rel_dir][ext] += 1
        if ext in (".txt", ".h5"):                 # remember these for later
            full = os.path.join(dirpath, fn)
            key_files.append((full, os.path.getsize(full)))

# Print the totals per file type
print("TOTAL FILES BY TYPE:")
for ext, n in ext_count.most_common():
    print(f"  {ext or '(no extension)'}: {n}")

# Print the number of files per folder
print("\nFILES PER FOLDER:")
for d, counter in sorted(dir_counts.items())[:40]:
    print(f"  {d}: {dict(counter)}")

# Print the .txt and .h5 files with their sizes in MB
print("\nKEY FILES (.txt and .h5):")
for path, size in sorted(key_files):
    print(f"  {path}  ({size/1e6:.1f} MB)")

# Helper: print the first few lines of a file, found by its name
def peek(filename, n_lines):
    matches = [p for p, _ in key_files if os.path.basename(p) == filename]
    if not matches:
        print(f"\n[{filename}] NOT FOUND")
        return
    print(f"\n===== {matches[0]} (first {n_lines} lines) =====")
    with open(matches[0], "r", errors="replace") as f:
        for i, line in enumerate(f):
            if i >= n_lines:
                break
            print(line.rstrip())

peek("feature_labels.txt", 40)   # the class label list
peek("h5_structure.txt", 60)     # description of the h5 graph layout
peek("train.txt", 5)             # first few training ids

TOTAL FILES BY TYPE:
  .step: 59665
  .txt: 6
  .h5: 3
  (no extension): 1
  .jpg: 1

FILES PER FOLDER:
  .: {'': 1}
  MFCAD++_dataset: {'.txt': 6, '.jpg': 1}
  MFCAD++_dataset/hierarchical_graphs: {'.h5': 3}
  MFCAD++_dataset/step/test: {'.step': 8949}
  MFCAD++_dataset/step/train: {'.step': 41766}
  MFCAD++_dataset/step/val: {'.step': 8950}

KEY FILES (.txt and .h5):
  /content/mfcad/MFCAD++_dataset/feature_labels.txt  (0.0 MB)
  /content/mfcad/MFCAD++_dataset/h5_structure.txt  (0.0 MB)
  /content/mfcad/MFCAD++_dataset/hierarchical_graphs/test_MFCAD++.h5  (254.9 MB)
  /content/mfcad/MFCAD++_dataset/hierarchical_graphs/training_MFCAD++.h5  (1185.3 MB)
  /content/mfcad/MFCAD++_dataset/hierarchical_graphs/val_MFCAD++.h5  (253.0 MB)
  /content/mfcad/MFCAD++_dataset/readme.txt  (0.0 MB)
  /content/mfcad/MFCAD++_dataset/test.txt  (0.1 MB)
  /content/mfcad/MFCAD++_dataset/train.txt  (0.2 MB)
  /content/mfcad/MFCAD++_dataset/val.txt  (0.1 MB)

===== /content/mfcad/MFCAD++_dataset/feature_lab

In [11]:
#chunk 1
#5
# Build a small summary of what we found, to store on Drive
summary = {
    "files_by_type": dict(ext_count),
    "files_per_folder": {d: dict(c) for d, c in dir_counts.items()},
    "key_files": [{"path": p, "mb": round(s / 1e6, 1)} for p, s in key_files],
}

# Save it as a JSON note in the project's notes folder
helpers.save_json(summary, "notes", "chunk1_inspection.json")

# Record that chunk 1 is finished
helpers.mark_done("chunk1_load_data", note="uploaded, unzipped, inspected")
helpers.show_progress()


Saved: /content/drive/MyDrive/mfcad_project/notes/chunk1_inspection.json
Marked done: chunk1_load_data
chunk0_setup: finished 2026-11-23 11:07:56  folders and helpers created
chunk1_load_data: finished 2026-11-23 11:45:40  uploaded, unzipped, inspected


In [12]:
#chunk 2
#1

# Mount Google Drive so Colab can see our project folder (accept the popup)
from google.colab import drive
drive.mount('/content/drive')

# Load our helper tools from the project folder
import sys
sys.path.append("/content/drive/MyDrive/mfcad_project")
import helpers

# If the session was reset, the unzipped data is gone: unzip again from Drive.
# The marker file tells us whether the unzip had fully finished.
ZIP_PATH = "/content/drive/MyDrive/mfcad_project/data/MFCAD_dataset.zip"
!test -f /content/mfcad/.unzipped_ok && echo "Data already unzipped." || (mkdir -p /content/mfcad && unzip -q -o "{ZIP_PATH}" -d /content/mfcad && touch /content/mfcad/.unzipped_ok && echo "Unzipped successfully.")

# Show which chunks are finished
helpers.show_progress()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Data already unzipped.
chunk0_setup: finished 2026-11-23 11:07:56  folders and helpers created
chunk1_load_data: finished 2026-11-23 11:45:40  uploaded, unzipped, inspected


In [13]:
# chunk 2
#2

ROOT = "/content/mfcad/MFCAD++_dataset"   # where the dataset lives

# Print a text file starting at a given line number, so we only see what is new
def print_from(filename, start_line):
    print(f"\n===== {filename} (from line {start_line}) =====")
    with open(f"{ROOT}/{filename}", "r", errors="replace") as f:
        for i, line in enumerate(f):
            if i >= start_line:           # skip lines we already saw
                print(line.rstrip())

print_from("h5_structure.txt", 58)        # the part that was cut off before
print_from("readme.txt", 0)               # the whole readme


===== h5_structure.txt (from line 58) =====

E_2_idx - idxs for edges in adjacency matrix relating to concave edges in B-Rep graph level
E_2_values - values of edges in adjacency matrix relating to concave edges in B-Rep graph level
E_2_shape - shape of adjacency matrix relating to concave edges in B-Rep graph level

E_3_idx - idxs for edges in adjacency matrix relating to smooth edges in B-Rep graph level
E_3_values - values of edges in adjacency matrix relating to smooth edges in B-Rep graph level
E_3_shape - shape of adjacency matrix relating to smooth edges in B-Rep graph level

facet_labels - labels per each facet in mesh graph level

A_4_idx - idxs for edges in incidence matrix between graph levels (Now redundant)
A_4_values - values of edges in incidence matrix between graph levels (Now redundant)
A_4_shape - shape of incidence matrix between graph levels (Now redundant)

===== readme.txt (from line 0) =====
# Description

- The MFCAD++ dataset is a machining feature recognitio

In [14]:
#chunk 2
#3

import h5py            # the library for reading .h5 files
import numpy as np     # numbers and arrays

H5_DIR = "/content/mfcad/MFCAD++_dataset/hierarchical_graphs"
VAL_H5 = f"{H5_DIR}/val_MFCAD++.h5"       # the smallest split, safest to explore

with h5py.File(VAL_H5, "r") as f:         # open the file for reading only
    batch_names = list(f.keys())          # each batch is a "group" in the file
    print("Number of batches in val:", len(batch_names))
    print("First 5 batch names:", batch_names[:5])

    # Look inside the first batch: every dataset with its shape and data type
    first = f[batch_names[0]]
    print(f"\nDatasets inside {batch_names[0]}:")
    for name in first.keys():
        ds = first[name]
        print(f"  {name:12s} shape={ds.shape}  dtype={ds.dtype}")

Number of batches in val: 312
First 5 batch names: ['0', '1', '10', '100', '101']

Datasets inside 0:
  A_1_idx      shape=(4567, 2)  dtype=int32
  A_1_shape    shape=(2,)  dtype=int32
  A_1_values   shape=(4567,)  dtype=float32
  A_2_idx      shape=(26908, 2)  dtype=int32
  A_2_shape    shape=(2,)  dtype=int32
  A_2_values   shape=(26908,)  dtype=float32
  A_3_idx      shape=(8971, 2)  dtype=int32
  A_3_shape    shape=(2,)  dtype=int32
  A_3_values   shape=(8971,)  dtype=float32
  A_4_idx      shape=(8971, 2)  dtype=int32
  A_4_shape    shape=(2,)  dtype=int32
  A_4_values   shape=(8971,)  dtype=float32
  CAD_model    shape=(29,)  dtype=|S5
  E_1_idx      shape=(3082, 2)  dtype=int32
  E_1_shape    shape=(2,)  dtype=int32
  E_1_values   shape=(3082,)  dtype=float32
  E_2_idx      shape=(1416, 2)  dtype=int32
  E_2_shape    shape=(2,)  dtype=int32
  E_2_values   shape=(1416,)  dtype=float32
  E_3_idx      shape=(69, 2)  dtype=int32
  E_3_shape    shape=(2,)  dtype=int32
  E_3_values   

In [15]:
#chunk 2
# 4
with h5py.File(VAL_H5, "r") as f:
    b = f[batch_names[0]]                 # the first batch

    # Print the first 5 entries of each dataset we care about
    for name in ["CAD_model", "idx", "V_1", "V_2", "labels",
                 "A_1_idx", "A_1_values", "A_1_shape"]:
        if name in b:
            print(f"\n{name} (first 5 entries):")
            print(b[name][:5])            # [:5] reads only 5 entries, never the whole array
        else:
            print(f"\n{name}: NOT in this batch")


CAD_model (first 5 entries):
[b'37958' b'38096' b'35710' b'1429' b'53736']

idx (first 5 entries):
[[  40  287]
 [  78  737]
 [ 122 1383]
 [ 161 1551]
 [ 188 1761]]

V_1 (first 5 entries):
[[0.7597973  0.9999995  0.56261206 0.40760347 0.09090901]
 [0.5367005  0.4995576  0.4740305  0.73571163 0.09090901]
 [0.04186805 0.9810386  0.737015   0.9844979  0.18181802]
 [0.46239087 0.49999976 0.23701525 0.4718845  0.09090901]
 [1.         0.49220538 0.9999996  0.50684977 0.09090901]]

V_2 (first 5 entries):
[[ 0.49379143  0.37082142  0.35747135  2.701897  ]
 [ 0.49208504  0.37082142  0.35747135  3.4445004 ]
 [ 0.5071415   0.37082142  0.35747135 -3.1079137 ]
 [ 0.5071373   0.37082142  0.35747135 -3.106109  ]
 [ 0.49284163  0.37082142  0.35747135  3.1152363 ]]

labels (first 5 entries):
[24.  8. 23.  8. 24.]

A_1_idx (first 5 entries):
[[0 1]
 [0 2]
 [0 3]
 [0 4]
 [0 5]]

A_1_values (first 5 entries):
[1. 1. 1. 1. 1.]

A_1_shape (first 5 entries):
[875 875]


In [16]:
#chunk 2
# 5
import re

# Read the label names (0 - Chamfer, 1 - Through hole, ...) from the labels file
label_names = {}
with open(f"{ROOT}/feature_labels.txt") as fh:
    for line in fh:
        m = re.match(r"^(\d+)\s*-\s*(.+)$", line.strip())   # matches "12 - Blind hole"
        if m:
            label_names[int(m.group(1))] = m.group(2)

counts = np.zeros(64, dtype=np.int64)     # room for class ids 0..63 (we expect 0..24)
total_faces = 0                           # running total of faces
total_models = 0                          # running total of CAD models

with h5py.File(VAL_H5, "r") as f:
    for name in f.keys():                 # go through every batch
        lab = f[name]["labels"][:]        # all labels in this batch
        if lab.ndim > 1 and lab.shape[1] > 1:
            lab = lab.argmax(axis=1)      # if labels are one-hot, turn into ids
        lab = lab.reshape(-1).astype(int) # make it a flat list of integers
        bc = np.bincount(lab)             # count how many faces per class id
        counts[:len(bc)] += bc
        total_faces += len(lab)
        if "CAD_model" in f[name]:
            total_models += f[name]["CAD_model"].shape[0]

print(f"Batches: {len(batch_names)}  |  CAD models: {total_models} (expect 8950)")
print(f"Total faces (labels): {total_faces}\n")
print(f"{'id':>3}  {'class':35s} {'faces':>9}  {'share':>6}")
for cid in np.nonzero(counts)[0]:
    share = 100 * counts[cid] / total_faces
    print(f"{cid:>3}  {label_names.get(int(cid), '?'):35s} {counts[cid]:>9}  {share:5.1f}%")

# Save the class table to Drive and mark chunk 2 done
helpers.save_json(
    {"total_faces": int(total_faces), "total_models": int(total_models),
     "class_counts": {int(c): int(counts[c]) for c in np.nonzero(counts)[0]}},
    "notes", "chunk2_val_class_counts.json")
helpers.mark_done("chunk2_understand_data", note="h5 structure and val class counts")

Batches: 312  |  CAD models: 8639 (expect 8950)
Total faces (labels): 259490

 id  class                                   faces   share
  0  Chamfer                                  3610    1.4%
  1  Through hole                             3641    1.4%
  2  Triangular passage                      10744    4.1%
  3  Rectangular passage                     14742    5.7%
  4  6-sided passage                         21739    8.4%
  5  Triangular through slot                  2461    0.9%
  6  Rectangular through slot                 4138    1.6%
  7  Circular through slot                    1298    0.5%
  8  Rectangular through step                 6954    2.7%
  9  2-sided through step                     9978    3.8%
 10  Slanted through step                     6905    2.7%
 11  O-ring                                   9550    3.7%
 12  Blind hole                               6564    2.5%
 13  Triangular pocket                       12550    4.8%
 14  Rectangular pocket              

In [17]:
#chunk 2b 1
import h5py
import numpy as np

ROOT = "/content/mfcad/MFCAD++_dataset"
VAL_H5 = f"{ROOT}/hierarchical_graphs/val_MFCAD++.h5"

# 1) The official list of validation ids from val.txt
with open(f"{ROOT}/val.txt") as fh:
    val_ids = [line.strip() for line in fh if line.strip()]
print("ids in val.txt:", len(val_ids), "| unique:", len(set(val_ids)))

h5_ids = []                 # every CAD model id found in the h5 file
surface_codes = set()       # distinct surface type codes (value * 11, rounded)
label_min, label_max = 99, -1
labels_all_whole = True     # are all labels whole numbers like 8.0?
edge_sum_ok = 0             # batches where convex+concave+smooth == total edges
symmetric_ok = 0            # batches where every edge has its reverse edge
rows_match_ok = 0           # batches where labels length == number of faces
idx_end_ok = 0              # batches where the last idx row == (faces, facets)
n_batches = 0

with h5py.File(VAL_H5, "r") as f:
    for name in sorted(f.keys(), key=int):        # numeric order, not text order
        b = f[name]
        n_batches += 1
        h5_ids += [x.decode() for x in b["CAD_model"][:]]

        v1 = b["V_1"][:]                          # face features
        lab = b["labels"][:]                      # face labels
        surface_codes.update(np.round(v1[:, 4] * 11).astype(int).tolist())
        label_min, label_max = min(label_min, lab.min()), max(label_max, lab.max())
        labels_all_whole &= bool(np.all(lab == np.round(lab)))
        rows_match_ok += int(len(lab) == v1.shape[0])

        # convex + concave + smooth edge counts should equal the total
        total = b["E_1_idx"].shape[0] + b["E_2_idx"].shape[0] + b["E_3_idx"].shape[0]
        edge_sum_ok += int(total == b["A_1_idx"].shape[0])

        # symmetric adjacency: for every edge (i, j), is (j, i) also present?
        a = b["A_1_idx"][:].astype(np.int64)
        n = int(b["A_1_shape"][0])
        fwd = a[:, 0] * n + a[:, 1]               # encode each edge as one number
        rev = a[:, 1] * n + a[:, 0]               # the same edge reversed
        symmetric_ok += int(np.isin(rev, fwd).all())

        # does the last idx row equal (number of faces, number of facets)?
        last = b["idx"][-1]
        idx_end_ok += int(last[0] == v1.shape[0] and last[1] == b["V_2"].shape[0])

print("\nbatches checked:", n_batches)
print("models in h5:", len(h5_ids), "| unique:", len(set(h5_ids)))
missing = sorted(set(val_ids) - set(h5_ids), key=int)
extra = set(h5_ids) - set(val_ids)
print("in val.txt but NOT in h5:", len(missing), "| first 10:", missing[:10])
print("in h5 but NOT in val.txt:", len(extra))
print("\nsurface type codes found:", sorted(surface_codes))
print("label range:", label_min, "to", label_max, "| all whole numbers:", labels_all_whole)
print("labels length == faces:", rows_match_ok, "of", n_batches)
print("edge types add up:", edge_sum_ok, "of", n_batches)
print("adjacency symmetric:", symmetric_ok, "of", n_batches)
print("idx last row == (faces, facets):", idx_end_ok, "of", n_batches)

# Save the findings and mark this check done
import helpers
helpers.save_json({
    "val_txt_ids": len(val_ids), "h5_models": len(h5_ids),
    "missing_from_h5": len(missing), "extra_in_h5": len(extra),
    "surface_codes": sorted(surface_codes),
    "symmetric_batches": symmetric_ok, "batches": n_batches,
}, "notes", "chunk2b_verification.json")
helpers.mark_done("chunk2b_verify_h5", note="val h5 consistency checks")

ids in val.txt: 8950 | unique: 8950

batches checked: 312
models in h5: 8639 | unique: 8639
in val.txt but NOT in h5: 311 | first 10: ['114', '194', '471', '482', '599', '1331', '1674', '2143', '2400', '3173']
in h5 but NOT in val.txt: 0

surface type codes found: [1, 2, 3, 5, 7, 8]
label range: 0.0 to 24.0 | all whole numbers: True
labels length == faces: 312 of 312
edge types add up: 303 of 312
adjacency symmetric: 312 of 312
idx last row == (faces, facets): 0 of 312
Saved: /content/drive/MyDrive/mfcad_project/notes/chunk2b_verification.json
Marked done: chunk2b_verify_h5


In [18]:
#chunk 2b 2
import h5py, os
import numpy as np
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components

ROOT = "/content/mfcad/MFCAD++_dataset"
H5_DIR = f"{ROOT}/hierarchical_graphs"
VAL_H5 = f"{H5_DIR}/val_MFCAD++.h5"

# ---------- PART 1: what does idx mean? Compare it with connected components ----------
print("PART 1: idx versus connected components")
with h5py.File(VAL_H5, "r") as f:
    for name in ["0", "1", "2"]:                    # look at three batches only
        b = f[name]
        idx = b["idx"][:]                           # the array we do not understand yet
        a = b["A_1_idx"][:]                         # face adjacency edges
        n_faces = int(b["A_1_shape"][0])
        n_facets = b["V_2"].shape[0]
        # build a sparse graph and count its disconnected pieces (= separate parts?)
        g = coo_matrix((np.ones(len(a)), (a[:, 0], a[:, 1])), shape=(n_faces, n_faces))
        n_comp, comp = connected_components(g, directed=False)
        sizes = np.bincount(comp)                   # faces per component
        print(f"\nbatch {name}: faces={n_faces}, facets={n_facets}, "
              f"models={len(b['CAD_model'])}, components={n_comp}")
        print("  idx first 4 rows:", idx[:4].tolist())
        print("  idx last 2 rows :", idx[-2:].tolist())
        print("  first 5 component sizes:", sizes[:5].tolist())

# ---------- PART 2: the 9 batches where edge sets do not add up ----------
print("\nPART 2: batches where convex+concave+smooth != total edges")
with h5py.File(VAL_H5, "r") as f:
    for name in sorted(f.keys(), key=int):
        b = f[name]
        parts = b["E_1_idx"].shape[0] + b["E_2_idx"].shape[0] + b["E_3_idx"].shape[0]
        total = b["A_1_idx"].shape[0]
        if parts != total:
            print(f"  batch {name}: sets={parts}, A_1={total}, difference={parts-total}")

# ---------- PART 3: are the 311 missing models different? (STEP file size) ----------
print("\nPART 3: STEP file size, present versus missing models")
with open(f"{ROOT}/val.txt") as fh:
    val_ids = [l.strip() for l in fh if l.strip()]
with h5py.File(VAL_H5, "r") as f:
    present = set()
    for name in f.keys():
        present.update(x.decode() for x in f[name]["CAD_model"][:])
miss = [i for i in val_ids if i not in present]
size = lambda i: os.path.getsize(f"{ROOT}/step/val/{i}.step") / 1e3   # file size in KB
s_pres = np.array([size(i) for i in val_ids if i in present])
s_miss = np.array([size(i) for i in miss])
print(f"  present: median {np.median(s_pres):.1f} KB, mean {s_pres.mean():.1f} KB")
print(f"  missing: median {np.median(s_miss):.1f} KB, mean {s_miss.mean():.1f} KB")

# ---------- PART 4: do train and test lose models too? ----------
print("\nPART 4: model counts in train and test h5 files")
for h5name, txt in [("training_MFCAD++.h5", "train.txt"), ("test_MFCAD++.h5", "test.txt")]:
    with open(f"{ROOT}/{txt}") as fh:
        n_txt = sum(1 for l in fh if l.strip())
    n_h5 = 0
    with h5py.File(f"{H5_DIR}/{h5name}", "r") as f:
        for name in f.keys():
            n_h5 += f[name]["CAD_model"].shape[0]   # read only the small id list
    print(f"  {h5name}: h5={n_h5}, {txt}={n_txt}, missing={n_txt - n_h5}")

PART 1: idx versus connected components

batch 0: faces=875, facets=8971, models=29, components=29
  idx first 4 rows: [[40, 287], [78, 737], [122, 1383], [161, 1551]]
  idx last 2 rows : [[839, 8654], [874, 8970]]
  first 5 component sizes: [41, 38, 44, 39, 27]

batch 1: faces=739, facets=8728, models=24, components=24
  idx first 4 rows: [[941, 9702], [969, 9806], [1004, 10302], [1031, 10530]]
  idx last 2 rows : [[1630, 18194], [1659, 18310]]
  first 5 component sizes: [21, 28, 35, 27, 19]

batch 2: faces=859, facets=8998, models=29, components=29
  idx first 4 rows: [[1744, 19370], [1769, 19486], [1800, 19890], [1839, 20034]]
  idx last 2 rows : [[2527, 27338], [2565, 28044]]
  first 5 component sizes: [38, 25, 31, 39, 32]

PART 2: batches where convex+concave+smooth != total edges
  batch 1: sets=3884, A_1=3882, difference=2
  batch 5: sets=5279, A_1=5277, difference=2
  batch 16: sets=4368, A_1=4366, difference=2
  batch 39: sets=4239, A_1=4237, difference=2
  batch 171: sets=507

In [19]:
#chunk 3
# Mount Google Drive (accept the popup) and load our helper tools
from google.colab import drive
drive.mount('/content/drive')
import sys
sys.path.append("/content/drive/MyDrive/mfcad_project")
import helpers

# If the session was reset, the unzipped data is gone: unzip again from Drive.
# The marker file tells us whether a previous unzip fully finished.
ZIP_PATH = "/content/drive/MyDrive/mfcad_project/data/MFCAD_dataset.zip"
!test -f /content/mfcad/.unzipped_ok && echo "Data already unzipped." || (mkdir -p /content/mfcad && unzip -q -o "{ZIP_PATH}" -d /content/mfcad && touch /content/mfcad/.unzipped_ok && echo "Unzipped successfully.")

helpers.show_progress()   # chunk0, chunk1, chunk2, chunk2b should be listed

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Data already unzipped.
chunk0_setup: finished 2026-11-23 11:07:56  folders and helpers created
chunk1_load_data: finished 2026-11-23 11:45:40  uploaded, unzipped, inspected
chunk2_understand_data: finished 2026-11-23 12:01:15  h5 structure and val class counts
chunk2b_verify_h5: finished 2026-11-23 12:22:47  val h5 consistency checks


In [20]:
# chunk 3
#1
# Mount Google Drive (accept the popup) and load our helper tools
from google.colab import drive
drive.mount('/content/drive')
import sys
sys.path.append("/content/drive/MyDrive/mfcad_project")
import helpers

# If the session was reset, the unzipped data is gone: unzip again from Drive.
# The marker file tells us whether a previous unzip fully finished.
ZIP_PATH = "/content/drive/MyDrive/mfcad_project/data/MFCAD_dataset.zip"
!test -f /content/mfcad/.unzipped_ok && echo "Data already unzipped." || (mkdir -p /content/mfcad && unzip -q -o "{ZIP_PATH}" -d /content/mfcad && touch /content/mfcad/.unzipped_ok && echo "Unzipped successfully.")

helpers.show_progress()   # chunk0, chunk1, chunk2, chunk2b should be listed

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Data already unzipped.
chunk0_setup: finished 2026-11-23 11:07:56  folders and helpers created
chunk1_load_data: finished 2026-11-23 11:45:40  uploaded, unzipped, inspected
chunk2_understand_data: finished 2026-11-23 12:01:15  h5 structure and val class counts
chunk2b_verify_h5: finished 2026-11-23 12:22:47  val h5 consistency checks


In [21]:
#chunk 3
#2

%%writefile /content/drive/MyDrive/mfcad_project/cache_builder.py
# cache_builder.py
# Converts one MFCAD++ h5 split into a small "face-level" cache that loads fast.
# It reads ONLY the face-level arrays (V_1, labels, A_1, E_1..E_3, CAD_model).
# The large mesh-level arrays are never read, so memory use stays low.

import os                                   # file paths and safe file replacing
import h5py                                 # reads the .h5 files
import numpy as np                          # array maths
import torch                                # we save the result with torch.save
from scipy.sparse import coo_matrix         # sparse graph, used to find separate parts
from scipy.sparse.csgraph import connected_components   # finds disconnected pieces

N_TYPES = 12      # surface type = round(value * 11), so codes 0..11 -> 12 slots
N_CLASSES = 25    # labels 0..24 (24 = Stock)


def edge_types(b, n_faces, a_idx):
    """For every edge in A_1, find whether it is convex (0), concave (1),
    smooth (2) or unknown (3, not found in any of the three lists)."""
    # Turn each (row, col) edge into ONE number so we can search quickly
    keys = a_idx[:, 0].astype(np.int64) * n_faces + a_idx[:, 1]
    order = np.argsort(keys)                    # sort the edge numbers
    sorted_keys = keys[order]
    etype = np.full(len(keys), 3, dtype=np.int8)   # start with "unknown" for all
    not_found = 0                               # edges in E lists but absent from A_1
    for t, name in enumerate(["E_1_idx", "E_2_idx", "E_3_idx"]):
        e = b[name][:]
        if e.size == 0:                         # no edges of this type in this batch
            continue
        e = e.reshape(-1, 2).astype(np.int64)
        ek = e[:, 0] * n_faces + e[:, 1]        # same one-number encoding
        pos = np.searchsorted(sorted_keys, ek)             # where would each edge sit?
        pos = np.clip(pos, 0, len(sorted_keys) - 1)        # keep positions in range
        found = sorted_keys[pos] == ek                     # does the edge really exist?
        etype[order[pos[found]]] = t                       # write its type
        not_found += int((~found).sum())
    return etype, not_found


def build_split(h5_path, out_path):
    """Read one h5 split batch by batch and save the face-level cache."""
    batches = []                                # one small dict per h5 batch
    class_counts = np.zeros(N_CLASSES, dtype=np.int64)
    n_models = n_faces_total = n_edges_total = 0
    unknown_edges = e_not_in_a1 = cc_mismatch = 0
    codes_seen = set()
    max_dev = 0.0                               # how far surface values are from whole codes

    with h5py.File(h5_path, "r") as f:
        names = sorted(f.keys(), key=int)       # numeric order: 0, 1, 2 ... not text order
        for k, name in enumerate(names):
            b = f[name]
            v1 = b["V_1"][:].astype(np.float32)             # face features
            y = b["labels"][:].astype(np.int64)             # face labels (whole numbers)
            a = b["A_1_idx"][:].astype(np.int64)            # face adjacency edges
            n = int(b["A_1_shape"][0])                      # number of faces in this batch
            ids = [x.decode() for x in b["CAD_model"][:]]   # CAD model ids

            # Safety checks: stop loudly if the file is not shaped as expected
            if v1.shape[0] != n or len(y) != n:
                raise ValueError(f"batch {name}: faces, features and labels disagree")
            if y.min() < 0 or y.max() >= N_CLASSES:
                raise ValueError(f"batch {name}: label outside 0..24")

            # Surface type: recover the integer code, then one-hot encode it
            code_f = v1[:, 4] * 11
            code = np.rint(code_f).astype(np.int64)
            max_dev = max(max_dev, float(np.abs(code_f - code).max()))
            if code.min() < 0 or code.max() >= N_TYPES:
                raise ValueError(f"batch {name}: surface code outside 0..11")
            codes_seen.update(code.tolist())
            onehot = np.eye(N_TYPES, dtype=np.float32)[code]
            x = np.concatenate([v1[:, :4], onehot], axis=1)  # 4 numbers + 12 one-hot = 16

            # Edge types (convex / concave / smooth / unknown)
            etype, nf = edge_types(b, n, a)
            unknown_edges += int((etype == 3).sum())
            e_not_in_a1 += nf

            # Which part (CAD model) does each face belong to?
            g = coo_matrix((np.ones(len(a), dtype=np.float32), (a[:, 0], a[:, 1])),
                           shape=(n, n))
            n_comp, comp = connected_components(g, directed=False)
            if n_comp != len(ids):                          # parts must equal model ids
                cc_mismatch += 1

            batches.append({
                "name": name,                                # batch name in the h5 file
                "x": torch.from_numpy(x),                    # (faces, 16) float32
                "y": torch.from_numpy(y.astype(np.int8)),    # (faces,) class ids
                "edge_index": torch.from_numpy(a.T.astype(np.int32)),  # (2, edges)
                "edge_type": torch.from_numpy(etype),        # (edges,) int8
                "face_model": torch.from_numpy(comp.astype(np.int16)), # part number per face
                "model_ids": ids,                            # ids of the parts, in order
            })
            class_counts += np.bincount(y, minlength=N_CLASSES)
            n_models += len(ids)
            n_faces_total += n
            n_edges_total += len(a)
            if (k + 1) % 200 == 0:
                print(f"  processed {k + 1} / {len(names)} batches")

    summary = {
        "n_batches": len(batches), "n_models": n_models,
        "n_faces": n_faces_total, "n_edges": n_edges_total,
        "class_counts": class_counts.tolist(),
        "unknown_edge_count": unknown_edges,
        "typed_edges_missing_from_A1": e_not_in_a1,
        "batches_where_components_!= models": cc_mismatch,
        "surface_codes_seen": sorted(codes_seen),
        "max_surface_code_deviation": round(max_dev, 6),
    }
    # Save to a temporary name first, then rename, so a crash never leaves a broken file
    torch.save({"batches": batches, "summary": summary}, out_path + ".tmp")
    os.replace(out_path + ".tmp", out_path)
    print("Saved:", out_path)
    return summary

Writing /content/drive/MyDrive/mfcad_project/cache_builder.py


In [22]:
#chunk 3
#3
import os, importlib
import cache_builder                       # our new file on Drive
importlib.reload(cache_builder)            # reload it in case we edited it

PROJECT = "/content/drive/MyDrive/mfcad_project"
H5_DIR = "/content/mfcad/MFCAD++_dataset/hierarchical_graphs"

for split, h5name in [("val", "val_MFCAD++.h5"), ("test", "test_MFCAD++.h5")]:
    out = f"{PROJECT}/data/cache_{split}.pt"
    if os.path.exists(out):                # already built: do not repeat the work
        print(f"{split}: cache already exists, skipping.")
        continue
    print(f"Building {split} cache...")
    s = cache_builder.build_split(f"{H5_DIR}/{h5name}", out)
    helpers.save_json(s, "notes", f"chunk3_summary_{split}.json")
    # Print the summary without the long class list
    print({k: v for k, v in s.items() if k != "class_counts"})

Building val cache...
  processed 200 / 312 batches
Saved: /content/drive/MyDrive/mfcad_project/data/cache_val.pt
Saved: /content/drive/MyDrive/mfcad_project/notes/chunk3_summary_val.json
{'n_batches': 312, 'n_models': 8639, 'n_faces': 259490, 'n_edges': 1336311, 'unknown_edge_count': 0, 'typed_edges_missing_from_A1': 0, 'batches_where_components_!= models': 30, 'surface_codes_seen': [1, 2, 3, 5, 7, 8], 'max_surface_code_deviation': 7e-06}
Building test cache...
  processed 200 / 317 batches
Saved: /content/drive/MyDrive/mfcad_project/data/cache_test.pt
Saved: /content/drive/MyDrive/mfcad_project/notes/chunk3_summary_test.json
{'n_batches': 317, 'n_models': 8632, 'n_faces': 259128, 'n_edges': 1334605, 'unknown_edge_count': 0, 'typed_edges_missing_from_A1': 0, 'batches_where_components_!= models': 18, 'surface_codes_seen': [1, 2, 3, 5, 7, 8], 'max_surface_code_deviation': 7e-06}


In [23]:
#chunk 3
# 4

out = f"{PROJECT}/data/cache_train.pt"
if os.path.exists(out):
    print("train: cache already exists, skipping.")
else:
    print("Building train cache (this can take several minutes)...")
    s = cache_builder.build_split(f"{H5_DIR}/training_MFCAD++.h5", out)
    helpers.save_json(s, "notes", "chunk3_summary_train.json")
    print({k: v for k, v in s.items() if k != "class_counts"})

# All three caches exist now: record chunk 3 as finished
if all(os.path.exists(f"{PROJECT}/data/cache_{s}.pt") for s in ["val", "test", "train"]):
    helpers.mark_done("chunk3_cache", note="face-level caches for train, val and test")
helpers.show_progress()

Building train cache (this can take several minutes)...
  processed 200 / 1473 batches
  processed 400 / 1473 batches
  processed 600 / 1473 batches
  processed 800 / 1473 batches
  processed 1000 / 1473 batches
  processed 1200 / 1473 batches
  processed 1400 / 1473 batches
Saved: /content/drive/MyDrive/mfcad_project/data/cache_train.pt
Saved: /content/drive/MyDrive/mfcad_project/notes/chunk3_summary_train.json
{'n_batches': 1473, 'n_models': 40292, 'n_faces': 1207577, 'n_edges': 6218239, 'unknown_edge_count': 0, 'typed_edges_missing_from_A1': 0, 'batches_where_components_!= models': 92, 'surface_codes_seen': [1, 2, 3, 5, 7, 8], 'max_surface_code_deviation': 7e-06}
Marked done: chunk3_cache
chunk0_setup: finished 2026-11-23 11:07:56  folders and helpers created
chunk1_load_data: finished 2026-11-23 11:45:40  uploaded, unzipped, inspected
chunk2_understand_data: finished 2026-11-23 12:01:15  h5 structure and val class counts
chunk2b_verify_h5: finished 2026-11-23 12:22:47  val h5 consi

In [24]:

#chunk 3
#5
import h5py
import numpy as np
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components

ROOT = "/content/mfcad/MFCAD++_dataset"

def boundary_report(h5_path, txt_path):
    # How many model ids the official list says there should be
    with open(txt_path) as fh:
        n_txt = sum(1 for line in fh if line.strip())
    n_models, gaps, prev_end = 0, [], -1
    with h5py.File(h5_path, "r") as f:
        names = sorted(f.keys(), key=int)
        for name in names:
            b = f[name]
            n_models += b["CAD_model"].shape[0]
            idx = b["idx"][:]                          # last-face index of each model
            a = b["A_1_idx"][:].astype(np.int64)
            n = int(b["A_1_shape"][0])
            g = coo_matrix((np.ones(len(a)), (a[:, 0], a[:, 1])), shape=(n, n))
            _, comp = connected_components(g, directed=False)
            first_size = int((comp == 0).sum())        # faces in the first model
            start = int(idx[0, 0]) - first_size + 1    # global index of this batch's first face
            gaps.append(start - (prev_end + 1))        # faces skipped since the last batch
            prev_end = int(idx[-1, 0])
    gaps = np.array(gaps)
    print(f"  batches: {len(names)} | models in h5: {n_models} | in txt: {n_txt} "
          f"| missing: {n_txt - n_models}")
    print(f"  gap before batch 0: {gaps[0]} faces")
    inner = gaps[1:]                                   # gaps between consecutive batches
    print(f"  gaps between batches: {len(inner)} | positive: {(inner > 0).sum()} "
          f"| zero: {(inner == 0).sum()} | negative: {(inner < 0).sum()}")
    print(f"  gap size (faces): min {inner.min()}, median {int(np.median(inner))}, "
          f"max {inner.max()}")

for h5name, txt in [("val_MFCAD++.h5", "val.txt"),
                    ("test_MFCAD++.h5", "test.txt"),
                    ("training_MFCAD++.h5", "train.txt")]:
    print(h5name)
    boundary_report(f"{ROOT}/hierarchical_graphs/{h5name}", f"{ROOT}/{txt}")

val_MFCAD++.h5
  batches: 312 | models in h5: 8639 | in txt: 8950 | missing: 311
  gap before batch 0: 0 faces
  gaps between batches: 311 | positive: 311 | zero: 0 | negative: 0
  gap size (faces): min 13, median 31, max 57
test_MFCAD++.h5
  batches: 317 | models in h5: 8632 | in txt: 8949 | missing: 317
  gap before batch 0: 0 faces
  gaps between batches: 316 | positive: 316 | zero: 0 | negative: 0
  gap size (faces): min 13, median 31, max 78
training_MFCAD++.h5
  batches: 1473 | models in h5: 40292 | in txt: 41766 | missing: 1474
  gap before batch 0: 0 faces
  gaps between batches: 1472 | positive: 1472 | zero: 0 | negative: 0
  gap size (faces): min 10, median 31, max 68


In [26]:
#chunk 3b
# 1

import sys  # lets us add folders to Python's search path
sys.path.append("/content/drive/MyDrive/mfcad_project")  # make helpers.py importable

import torch  # to load the saved cache
import h5py  # to read the h5 file
import numpy as np  # for array maths

# Load the val cache saved in chunk 3 (our own file, so full loading is fine)
cache = torch.load("/content/drive/MyDrive/mfcad_project/data/cache_val.pt", weights_only=False)  # dict with "batches"
b = cache["batches"][11]  # the failing batch found in the last run (index 11)

# Convert the part number per face from a torch tensor to a NumPy integer array
fm = b["face_model"].numpy().astype(int)  # one part number per face

print("batch name:", b["name"])  # group name in the h5 file
print("parts from components:", int(fm.max()) + 1)  # what my code found
print("model ids listed:", len(b["model_ids"]))  # what the file lists
print("faces in batch:", len(b["y"]))  # number of faces

# Part sizes according to my connected-component numbering
print("sizes (components):", np.bincount(fm).tolist())  # faces per found part

# Part sizes according to idx in the h5 file (idx[:,0] = last face of each model)
with h5py.File("/content/mfcad/MFCAD++_dataset/hierarchical_graphs/val_MFCAD++.h5", "r") as f:  # open val h5
    idx = f[str(b["name"])]["idx"][:]  # idx array for this batch
last = idx[:, 0].astype(int)  # global index of each model's last face
start = last[-1] - len(b["y"]) + 1  # global index of this batch's first face
edges = np.concatenate([[start - 1], last])  # boundaries between models
print("sizes (idx):", np.diff(edges).tolist())  # faces per model from idx

batch name: 11
parts from components: 29
model ids listed: 28
faces in batch: 770
sizes (components): [21, 18, 8, 33, 27, 34, 32, 24, 44, 31, 20, 26, 34, 17, 11, 25, 29, 27, 28, 27, 24, 33, 22, 38, 12, 28, 34, 38, 25]
sizes (idx): [21, 26, 33, 27, 34, 32, 24, 44, 31, 20, 26, 34, 17, 11, 25, 29, 27, 28, 27, 24, 33, 22, 38, 12, 28, 34, 38, 25]


In [27]:
#chunk 3b
# 2

import sys  # lets us add folders to Python's search path
sys.path.append("/content/drive/MyDrive/mfcad_project")  # make helpers.py importable

import os  # for safe file replacing
import torch  # to load and save the caches
import h5py  # to read the h5 files
import numpy as np  # for array maths
import helpers  # our saving/progress helpers

DATA = "/content/drive/MyDrive/mfcad_project/data"  # folder with the caches
H5 = "/content/mfcad/MFCAD++_dataset/hierarchical_graphs"  # folder with the h5 files

# Pairs of (cache file, h5 file), using the names printed in chunk 3
jobs = [("cache_val.pt", "val_MFCAD++.h5"),  # validation split
        ("cache_test.pt", "test_MFCAD++.h5"),  # test split
        ("cache_train.pt", "training_MFCAD++.h5")]  # training split

for cache_name, h5_name in jobs:  # go through the three splits
    path = f"{DATA}/{cache_name}"  # full path of the cache
    cache = torch.load(path, weights_only=False)  # load our own cache
    changed = 0  # counts batches whose numbering changed
    with h5py.File(f"{H5}/{h5_name}", "r") as f:  # open the matching h5 file
        for b in cache["batches"]:  # go through every batch
            n_faces = len(b["y"])  # faces in this batch
            last = f[str(b["name"])]["idx"][:, 0].astype(np.int64)  # last face of each model (global)
            start = last[-1] - n_faces + 1  # global index of the batch's first face
            sizes = np.diff(np.concatenate([[start - 1], last]))  # faces per model
            # Safety checks: stop with an error instead of saving wrong data
            assert sizes.min() > 0, "a model has zero or negative size"  # sizes must be positive
            assert sizes.sum() == n_faces, "sizes do not add up to faces"  # must cover all faces
            assert len(sizes) == len(b["model_ids"]), "idx count != model ids"  # one id per model
            new_fm = np.repeat(np.arange(len(sizes)), sizes)  # part number for every face
            old_fm = b["face_model"].numpy().astype(int)  # the old connected-component numbering
            if not np.array_equal(new_fm, old_fm):  # did the numbering change?
                changed += 1  # count it
            b["face_model_cc"] = b["face_model"]  # keep the old numbering under another name
            b["face_model"] = torch.from_numpy(new_fm.astype(np.int16))  # store the new numbering
    torch.save(cache, path + ".tmp")  # save to a temporary file first
    os.replace(path + ".tmp", path)  # then replace the old cache safely
    print(cache_name, "| batches:", len(cache["batches"]), "| numbering changed in:", changed)  # report

helpers.mark_done("chunk3b_face_model_fix", "face_model rebuilt from idx in all caches")  # record progress

cache_val.pt | batches: 312 | numbering changed in: 30
cache_test.pt | batches: 317 | numbering changed in: 18
cache_train.pt | batches: 1473 | numbering changed in: 92
Marked done: chunk3b_face_model_fix


In [1]:
#checkpoint 1: checking if the files are still in memory after reconnecting

import os  # for checking files and folders
import sys  # lets us add folders to Python's search path
import shutil  # for disk space information

# Reconnect Google Drive (a permission pop-up may appear; accept it with your main account)
from google.colab import drive  # import the Colab Drive helper
drive.mount('/content/drive')  # connect Google Drive (safe to run twice)

PROJECT = "/content/drive/MyDrive/mfcad_project"  # our project folder on Drive
sys.path.append(PROJECT)  # make helpers.py importable

# 1) Check which of our important Drive files still exist
print("=== Files on Drive ===")  # section title
for name in ["helpers.py", "cache_builder.py", "progress.json",  # code and progress files
             "data/MFCAD_dataset.zip", "data/cache_train.pt",  # dataset zip and train cache
             "data/cache_val.pt", "data/cache_test.pt"]:  # val and test caches
    path = f"{PROJECT}/{name}"  # full path of the file
    if os.path.exists(path):  # does the file exist?
        size_mb = os.path.getsize(path) / 1e6  # size in megabytes
        print(f"OK       {name}  ({size_mb:.0f} MB)")  # show it exists with its size
    else:  # file not found
        print(f"MISSING  {name}")  # show it is missing

# 2) List the saved folders (models, results, notes)
print("\n=== Folder contents ===")  # section title
for folder in ["models", "results", "notes"]:  # the three subfolders
    path = f"{PROJECT}/{folder}"  # full path of the folder
    items = sorted(os.listdir(path)) if os.path.isdir(path) else None  # list files or None
    print(folder, "->", items if items is not None else "folder not found")  # show the result

# 3) Show our progress record (which chunks are finished)
print("\n=== Progress ===")  # section title
try:  # try, in case helpers.py is missing
    import helpers  # our own helper module
    helpers.show_progress()  # print finished chunks
except Exception as e:  # if something goes wrong
    print("Could not read progress:", e)  # show the reason

# 4) Check whether the unzipped dataset still exists in the temporary Colab disk
print("\n=== Temporary Colab disk ===")  # section title
print("Unzip marker present:", os.path.exists("/content/mfcad/.unzipped_ok"))  # True = dataset still unzipped
print("h5 folder present:", os.path.isdir("/content/mfcad/MFCAD++_dataset/hierarchical_graphs"))  # h5 files there?

# 5) Check the machine itself (RAM, disk, CPU/GPU)
import torch  # to check the device
import psutil  # to read memory information
ram = psutil.virtual_memory()  # current RAM status
print("\n=== Machine ===")  # section title
print(f"RAM used: {ram.used/1e9:.1f} GB of {ram.total/1e9:.1f} GB")  # RAM usage
total, used, free = shutil.disk_usage("/content")  # temporary disk usage
print(f"Colab disk free: {free/1e9:.0f} GB")  # free temporary disk
print("PyTorch:", torch.__version__, "| GPU available:", torch.cuda.is_available())  # should be CPU

Mounted at /content/drive
=== Files on Drive ===
OK       helpers.py  (0 MB)
OK       cache_builder.py  (0 MB)
OK       progress.json  (0 MB)
OK       data/MFCAD_dataset.zip  (1616 MB)
OK       data/cache_train.pt  (142 MB)
OK       data/cache_val.pt  (31 MB)
OK       data/cache_test.pt  (31 MB)

=== Folder contents ===
models -> []
results -> []
notes -> ['chunk1_inspection.json', 'chunk2_val_class_counts.json', 'chunk2b_verification.json', 'chunk3_summary_test.json', 'chunk3_summary_train.json', 'chunk3_summary_val.json', 'test_note.json']

=== Progress ===
chunk0_setup: finished 2026-11-23 11:07:56  folders and helpers created
chunk1_load_data: finished 2026-11-23 11:45:40  uploaded, unzipped, inspected
chunk2_understand_data: finished 2026-11-23 12:01:15  h5 structure and val class counts
chunk2b_verify_h5: finished 2026-11-23 12:22:47  val h5 consistency checks
chunk3_cache: finished 2026-11-23 13:55:57  face-level caches for train, val and test
chunk3b_face_model_fix: finished 20

In [2]:
#chunk 4: a small MLP

import os  # for file paths and safe replacing
import sys  # lets us add folders to Python's search path
import time  # to measure how long each epoch takes
import torch  # main library for the neural network
import torch.nn as nn  # building blocks for networks

# Reconnect Drive (safe to run twice; accept the pop-up if it appears)
from google.colab import drive  # Colab Drive helper
drive.mount('/content/drive')  # connect Google Drive

PROJECT = "/content/drive/MyDrive/mfcad_project"  # our project folder
sys.path.append(PROJECT)  # make helpers.py importable
import helpers  # our saving/progress helpers

torch.manual_seed(0)  # fixed random seed so results can be repeated
NUM_CLASSES = 25  # labels 0-24
EPOCHS = 20  # how many passes over the training data
BATCH_SIZE = 4096  # faces per training step
LR = 1e-3  # learning rate

def load_xy(name):  # load one cache and stack all its batches into two big tensors
    cache = torch.load(f"{PROJECT}/data/{name}", weights_only=False)  # our own cache file
    x = torch.cat([torch.as_tensor(b["x"]) for b in cache["batches"]]).float()  # (faces,16) inputs
    y = torch.cat([torch.as_tensor(b["y"]) for b in cache["batches"]]).long()  # (faces,) class ids
    return x, y  # give both back

xtr, ytr = load_xy("cache_train.pt")  # training faces
xva, yva = load_xy("cache_val.pt")  # validation faces
print("train:", tuple(xtr.shape), "| val:", tuple(xva.shape))  # check shapes against the handoff

# A small MLP: 16 inputs -> 128 -> 128 -> 25 class scores
model = nn.Sequential(
    nn.Linear(16, 128), nn.ReLU(),  # first layer + non-linearity
    nn.Linear(128, 128), nn.ReLU(),  # second layer + non-linearity
    nn.Linear(128, NUM_CLASSES))  # output: one score per class
opt = torch.optim.Adam(model.parameters(), lr=LR)  # optimizer
loss_fn = nn.CrossEntropyLoss()  # standard classification loss

def evaluate(x, y):  # compute accuracy, macro-F1 and per-class IoU
    model.eval()  # switch to evaluation mode
    with torch.no_grad():  # no gradients needed
        pred = model(x).argmax(dim=1)  # predicted class per face
    cm = torch.bincount(y * NUM_CLASSES + pred, minlength=NUM_CLASSES**2).reshape(NUM_CLASSES, NUM_CLASSES)  # confusion matrix (rows = true)
    tp = cm.diag().float()  # correct faces per class
    fp = cm.sum(0).float() - tp  # wrongly predicted as this class
    fn = cm.sum(1).float() - tp  # faces of this class that were missed
    iou = tp / (tp + fp + fn).clamp(min=1)  # per-class IoU
    f1 = 2 * tp / (2 * tp + fp + fn).clamp(min=1)  # per-class F1
    acc = tp.sum() / cm.sum()  # overall accuracy
    return acc.item(), f1.mean().item(), iou, f1, cm  # results

# Resume from a checkpoint if one exists on Drive
CKPT = f"{PROJECT}/models/mlp_ckpt.pt"  # checkpoint path
start_epoch, history = 0, []  # defaults for a fresh start
if os.path.exists(CKPT):  # did an earlier run save something?
    ck = torch.load(CKPT, weights_only=False)  # load it
    model.load_state_dict(ck["model"])  # restore weights
    opt.load_state_dict(ck["opt"])  # restore optimizer state
    start_epoch, history = ck["epoch"], ck["history"]  # restore progress
    print("Resuming from epoch", start_epoch)  # tell the user

for epoch in range(start_epoch, EPOCHS):  # training loop
    t0 = time.time()  # start timer
    model.train()  # switch to training mode
    perm = torch.randperm(len(xtr))  # shuffle the faces
    total = 0.0  # running loss
    for i in range(0, len(xtr), BATCH_SIZE):  # go through mini-batches
        idx = perm[i:i + BATCH_SIZE]  # faces of this step
        opt.zero_grad()  # clear old gradients
        loss = loss_fn(model(xtr[idx]), ytr[idx])  # compute loss
        loss.backward()  # compute gradients
        opt.step()  # update weights
        total += loss.item() * len(idx)  # add to running loss
    acc, mf1, _, _, _ = evaluate(xva, yva)  # check on validation data
    history.append({"epoch": epoch + 1, "train_loss": total / len(xtr), "val_acc": acc, "val_macro_f1": mf1})  # log it
    print(f"epoch {epoch+1}/{EPOCHS} loss {total/len(xtr):.4f} val_acc {acc:.4f} val_macroF1 {mf1:.4f} ({time.time()-t0:.0f}s)")  # progress line
    torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "epoch": epoch + 1, "history": history}, CKPT + ".tmp")  # save to temp file
    os.replace(CKPT + ".tmp", CKPT)  # replace old checkpoint safely

# Final validation results with per-class detail
acc, mf1, iou, f1, cm = evaluate(xva, yva)  # evaluate the finished model
print(f"\nFINAL val accuracy {acc:.4f} | macro-F1 {mf1:.4f} | mean IoU {iou.mean():.4f}")  # summary
print("per-class IoU:", [round(v, 3) for v in iou.tolist()])  # IoU for classes 0-24
helpers.save_json({"val_accuracy": acc, "val_macro_f1": mf1, "val_mean_iou": iou.mean().item(),  # save results
                   "per_class_iou": iou.tolist(), "per_class_f1": f1.tolist(),
                   "confusion_matrix": cm.tolist(), "history": history}, "results", "chunk4_mlp_val.json")
helpers.mark_done("chunk4_mlp", "MLP baseline trained and evaluated on val")  # record progress

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
train: (1207577, 16) | val: (259490, 16)
epoch 1/20 loss 2.2701 val_acc 0.3766 val_macroF1 0.1276 (5s)
epoch 2/20 loss 1.9020 val_acc 0.4061 val_macroF1 0.1779 (5s)
epoch 3/20 loss 1.8469 val_acc 0.4127 val_macroF1 0.1836 (6s)
epoch 4/20 loss 1.8283 val_acc 0.4140 val_macroF1 0.1808 (4s)
epoch 5/20 loss 1.8150 val_acc 0.4163 val_macroF1 0.1885 (4s)
epoch 6/20 loss 1.8026 val_acc 0.4187 val_macroF1 0.1875 (6s)
epoch 7/20 loss 1.7903 val_acc 0.4210 val_macroF1 0.1922 (4s)
epoch 8/20 loss 1.7792 val_acc 0.4233 val_macroF1 0.1957 (4s)
epoch 9/20 loss 1.7692 val_acc 0.4240 val_macroF1 0.1999 (6s)
epoch 10/20 loss 1.7596 val_acc 0.4278 val_macroF1 0.1959 (4s)
epoch 11/20 loss 1.7488 val_acc 0.4315 val_macroF1 0.2038 (4s)
epoch 12/20 loss 1.7372 val_acc 0.4354 val_macroF1 0.2080 (6s)
epoch 13/20 loss 1.7272 val_acc 0.4376 val_macroF1 0.2118 (4s)
epoch 14/20 loss 1.7

In [6]:
#chunk 5:
#1
import os  # file paths and safe replacing
import sys  # lets us add folders to Python's search path
import time  # to time each epoch
import torch  # main library
import torch.nn as nn  # network building blocks

from google.colab import drive  # Colab Drive helper
drive.mount('/content/drive')  # connect Drive (safe to run twice)
PROJECT = "/content/drive/MyDrive/mfcad_project"  # project folder
sys.path.append(PROJECT)  # make helpers.py importable
import helpers  # our saving/progress helpers

TEST_RUN = False # True = quick check on a subset; set to False for the full run
torch.manual_seed(0)  # fixed seed for repeatable results
NUM_CLASSES = 25  # labels 0-24
HIDDEN = 64  # size of the face representation
LAYERS = 4  # how many neighbour-passing rounds (each sees one step further)
GROUP = 8  # h5 batches merged into one training step
LR = 2e-3  # learning rate
EPOCHS = 2 if TEST_RUN else 15  # passes over the training data
CKPT = f"{PROJECT}/models/{'gnn_test_ckpt.pt' if TEST_RUN else 'gnn_ckpt.pt'}"  # checkpoint file

def build_groups(batches):  # merge several batches into groups and prepare edge lists
    groups = []  # result list
    for s in range(0, len(batches), GROUP):  # step through batches in blocks
        off, xs, ys, eis, ets = 0, [], [], [], []  # node offset and collectors
        for b in batches[s:s + GROUP]:  # each batch in this block
            n = len(b["y"])  # faces in this batch
            xs.append(torch.as_tensor(b["x"]).float())  # inputs (n,16)
            ys.append(torch.as_tensor(b["y"]).long())  # class ids
            eis.append(torch.as_tensor(b["edge_index"]).long() + off)  # edges, shifted to global numbering
            ets.append(torch.as_tensor(b["edge_type"]).long())  # edge types 0/1/2
            off += n  # next batch starts after these faces
        x, y = torch.cat(xs), torch.cat(ys)  # all faces of the group
        ei, et = torch.cat(eis, dim=1), torch.cat(ets)  # all edges of the group
        rel = []  # one entry per edge type
        for t in range(3):  # convex, concave, smooth
            m = et == t  # edges of this type
            src, dst = ei[0][m], ei[1][m]  # sender and receiver faces
            deg = torch.bincount(dst, minlength=off).float().clamp(min=1)  # neighbours per face (for the mean)
            rel.append((src, dst, 1.0 / deg))  # store with 1/degree
        groups.append((x, y, rel))  # keep this group
    return groups  # done

def load_groups(name, limit):  # load a cache and build groups
    cache = torch.load(f"{PROJECT}/data/{name}", weights_only=False)  # our own cache
    batches = cache["batches"][:limit] if limit else cache["batches"]  # optional subset
    return build_groups(batches)  # merged groups

gtr = load_groups("cache_train.pt", 100 if TEST_RUN else None)  # training groups
gva = load_groups("cache_val.pt", 30 if TEST_RUN else None)  # validation groups
print("train groups:", len(gtr), "| val groups:", len(gva))  # sanity check

class Layer(nn.Module):  # one round of "look at neighbours"
    def __init__(self, d):  # d = size of face representation
        super().__init__()  # standard setup
        self.self_lin = nn.Linear(d, d)  # uses the face's own representation
        self.nbr_lin = nn.ModuleList([nn.Linear(d, d, bias=False) for _ in range(3)])  # one per edge type
    def forward(self, h, rel):  # h = (faces, d)
        out = self.self_lin(h)  # start with the face itself
        for t, (src, dst, invdeg) in enumerate(rel):  # each edge type
            agg = torch.zeros_like(h).index_add_(0, dst, h[src]) * invdeg.unsqueeze(1)  # mean of neighbours' values
            out = out + self.nbr_lin[t](agg)  # add transformed neighbour information
        return h + torch.relu(out)  # residual connection keeps earlier information

class Net(nn.Module):  # the full model
    def __init__(self):  # build the parts
        super().__init__()  # standard setup
        self.inp = nn.Linear(16, HIDDEN)  # 16 input features -> hidden size
        self.layers = nn.ModuleList([Layer(HIDDEN) for _ in range(LAYERS)])  # neighbour rounds
        self.out = nn.Linear(HIDDEN, NUM_CLASSES)  # hidden -> 25 class scores
    def forward(self, x, rel):  # run the model
        h = torch.relu(self.inp(x))  # embed the input features
        for layer in self.layers:  # each round
            h = layer(h, rel)  # mix in neighbour information
        return self.out(h)  # class scores per face

model = Net()  # create the model
opt = torch.optim.Adam(model.parameters(), lr=LR)  # optimizer
loss_fn = nn.CrossEntropyLoss()  # classification loss

def evaluate(groups):  # accuracy, macro-F1, per-class IoU on given groups
    model.eval()  # evaluation mode
    cm = torch.zeros(NUM_CLASSES, NUM_CLASSES, dtype=torch.long)  # confusion matrix (rows = true)
    with torch.no_grad():  # no gradients needed
        for x, y, rel in groups:  # each group
            pred = model(x, rel).argmax(1)  # predicted classes
            cm += torch.bincount(y * NUM_CLASSES + pred, minlength=NUM_CLASSES**2).reshape(NUM_CLASSES, NUM_CLASSES)  # add counts
    tp = cm.diag().float()  # correct per class
    fp = cm.sum(0).float() - tp  # false alarms per class
    fn = cm.sum(1).float() - tp  # misses per class
    iou = tp / (tp + fp + fn).clamp(min=1)  # per-class IoU
    f1 = 2 * tp / (2 * tp + fp + fn).clamp(min=1)  # per-class F1
    return (tp.sum() / cm.sum()).item(), f1.mean().item(), iou, f1, cm  # results

start_epoch, history = 0, []  # fresh start by default
if os.path.exists(CKPT):  # resume if a checkpoint exists
    ck = torch.load(CKPT, weights_only=False)  # load it
    model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"])  # restore weights and optimizer
    start_epoch, history = ck["epoch"], ck["history"]  # restore progress
    print("Resuming from epoch", start_epoch)  # tell the user

for epoch in range(start_epoch, EPOCHS):  # training loop
    t0 = time.time()  # start timer
    model.train()  # training mode
    total, nfaces = 0.0, 0  # running loss and face count
    for gi in torch.randperm(len(gtr)).tolist():  # groups in random order
        x, y, rel = gtr[gi]  # one group
        opt.zero_grad()  # clear gradients
        loss = loss_fn(model(x, rel), y)  # forward pass and loss
        loss.backward()  # gradients
        opt.step()  # update weights
        total += loss.item() * len(y); nfaces += len(y)  # accumulate loss
    acc, mf1, _, _, _ = evaluate(gva)  # validation check
    history.append({"epoch": epoch + 1, "train_loss": total / nfaces, "val_acc": acc, "val_macro_f1": mf1})  # log
    print(f"epoch {epoch+1}/{EPOCHS} loss {total/nfaces:.4f} val_acc {acc:.4f} val_macroF1 {mf1:.4f} ({time.time()-t0:.0f}s)")  # progress
    torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "epoch": epoch + 1, "history": history}, CKPT + ".tmp")  # temp save
    os.replace(CKPT + ".tmp", CKPT)  # safe replace

acc, mf1, iou, f1, cm = evaluate(gva)  # final evaluation
print(f"\nFINAL val accuracy {acc:.4f} | macro-F1 {mf1:.4f} | mean IoU {iou.mean():.4f}")  # summary
print("per-class IoU:", [round(v, 3) for v in iou.tolist()])  # IoU for classes 0-24
if not TEST_RUN:  # only save official results for the full run
    helpers.save_json({"val_accuracy": acc, "val_macro_f1": mf1, "val_mean_iou": iou.mean().item(),
                       "per_class_iou": iou.tolist(), "per_class_f1": f1.tolist(),
                       "confusion_matrix": cm.tolist(), "history": history}, "results", "chunk5_gnn_val.json")  # save to Drive
    helpers.mark_done("chunk5_gnn", "neighbour-aware GNN trained and evaluated on val")  # record progress
else:
    print("Quick test finished. Check the time per epoch, then set TEST_RUN = False.")  # next step hint
TEST_RUN = False  # False = full run on all data (15 epochs, saves results to Drive)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
train groups: 185 | val groups: 39
epoch 1/15 loss 1.3545 val_acc 0.6365 val_macroF1 0.4965 (24s)
epoch 2/15 loss 0.9429 val_acc 0.6739 val_macroF1 0.5785 (24s)
epoch 3/15 loss 0.8414 val_acc 0.7104 val_macroF1 0.6230 (25s)
epoch 4/15 loss 0.7644 val_acc 0.7329 val_macroF1 0.6667 (24s)
epoch 5/15 loss 0.7073 val_acc 0.7504 val_macroF1 0.6972 (24s)
epoch 6/15 loss 0.6656 val_acc 0.7444 val_macroF1 0.7062 (25s)
epoch 7/15 loss 0.6376 val_acc 0.7672 val_macroF1 0.7219 (25s)
epoch 8/15 loss 0.6141 val_acc 0.7669 val_macroF1 0.7292 (25s)
epoch 9/15 loss 0.5980 val_acc 0.7685 val_macroF1 0.7312 (24s)
epoch 10/15 loss 0.5802 val_acc 0.7781 val_macroF1 0.7373 (25s)
epoch 11/15 loss 0.5660 val_acc 0.7889 val_macroF1 0.7554 (25s)
epoch 12/15 loss 0.5507 val_acc 0.7917 val_macroF1 0.7556 (24s)
epoch 13/15 loss 0.5365 val_acc 0.8013 val_macroF1 0.7713 (25s)
epoch 14/15 l

In [11]:
#chunk 5:
#2
import os  # file paths and safe replacing
import sys  # lets us add folders to Python's search path
import time  # to time each epoch
import torch  # main library
import torch.nn as nn  # network building blocks

from google.colab import drive  # Colab Drive helper
drive.mount('/content/drive')  # connect Drive (safe to run twice)
PROJECT = "/content/drive/MyDrive/mfcad_project"  # project folder
sys.path.append(PROJECT)  # make helpers.py importable
import helpers  # our saving/progress helpers

TEST_RUN = False # True = quick check on a subset; set to False for the full run
torch.manual_seed(0)  # fixed seed for repeatable results
NUM_CLASSES = 25  # labels 0-24
HIDDEN = 128  # size of the face representation
LAYERS = 6  # how many neighbour-passing rounds (each sees one step further)
GROUP = 8  # h5 batches merged into one training step
LR = 2e-3  # learning rate
EPOCHS = 2 if TEST_RUN else 30 # passes over the training data
CKPT = f"{PROJECT}/models/{'gnn_test_ckpt.pt' if TEST_RUN else 'gnn_v2_ckpt.pt'}"  # checkpoint file

def build_groups(batches):  # merge several batches into groups and prepare edge lists
    groups = []  # result list
    for s in range(0, len(batches), GROUP):  # step through batches in blocks
        off, xs, ys, eis, ets = 0, [], [], [], []  # node offset and collectors
        for b in batches[s:s + GROUP]:  # each batch in this block
            n = len(b["y"])  # faces in this batch
            xs.append(torch.as_tensor(b["x"]).float())  # inputs (n,16)
            ys.append(torch.as_tensor(b["y"]).long())  # class ids
            eis.append(torch.as_tensor(b["edge_index"]).long() + off)  # edges, shifted to global numbering
            ets.append(torch.as_tensor(b["edge_type"]).long())  # edge types 0/1/2
            off += n  # next batch starts after these faces
        x, y = torch.cat(xs), torch.cat(ys)  # all faces of the group
        ei, et = torch.cat(eis, dim=1), torch.cat(ets)  # all edges of the group
        rel = []  # one entry per edge type
        for t in range(3):  # convex, concave, smooth
            m = et == t  # edges of this type
            src, dst = ei[0][m], ei[1][m]  # sender and receiver faces
            deg = torch.bincount(dst, minlength=off).float().clamp(min=1)  # neighbours per face (for the mean)
            rel.append((src, dst, 1.0 / deg))  # store with 1/degree
        groups.append((x, y, rel))  # keep this group
    return groups  # done

def load_groups(name, limit):  # load a cache and build groups
    cache = torch.load(f"{PROJECT}/data/{name}", weights_only=False)  # our own cache
    batches = cache["batches"][:limit] if limit else cache["batches"]  # optional subset
    return build_groups(batches)  # merged groups

gtr = load_groups("cache_train.pt", 100 if TEST_RUN else None)  # training groups
gva = load_groups("cache_val.pt", 30 if TEST_RUN else None)  # validation groups
print("train groups:", len(gtr), "| val groups:", len(gva))  # sanity check

class Layer(nn.Module):  # one round of "look at neighbours"
    def __init__(self, d):  # d = size of face representation
        super().__init__()  # standard setup
        self.self_lin = nn.Linear(d, d)  # uses the face's own representation
        self.nbr_lin = nn.ModuleList([nn.Linear(d, d, bias=False) for _ in range(3)])  # one per edge type
    def forward(self, h, rel):  # h = (faces, d)
        out = self.self_lin(h)  # start with the face itself
        for t, (src, dst, invdeg) in enumerate(rel):  # each edge type
            agg = torch.zeros_like(h).index_add_(0, dst, h[src]) * invdeg.unsqueeze(1)  # mean of neighbours' values
            out = out + self.nbr_lin[t](agg)  # add transformed neighbour information
        return h + torch.relu(out)  # residual connection keeps earlier information

class Net(nn.Module):  # the full model
    def __init__(self):  # build the parts
        super().__init__()  # standard setup
        self.inp = nn.Linear(16, HIDDEN)  # 16 input features -> hidden size
        self.layers = nn.ModuleList([Layer(HIDDEN) for _ in range(LAYERS)])  # neighbour rounds
        self.out = nn.Linear(HIDDEN, NUM_CLASSES)  # hidden -> 25 class scores
    def forward(self, x, rel):  # run the model
        h = torch.relu(self.inp(x))  # embed the input features
        for layer in self.layers:  # each round
            h = layer(h, rel)  # mix in neighbour information
        return self.out(h)  # class scores per face

model = Net()  # create the model
opt = torch.optim.Adam(model.parameters(), lr=LR)  # optimizer
loss_fn = nn.CrossEntropyLoss()  # classification loss

def evaluate(groups):  # accuracy, macro-F1, per-class IoU on given groups
    model.eval()  # evaluation mode
    cm = torch.zeros(NUM_CLASSES, NUM_CLASSES, dtype=torch.long)  # confusion matrix (rows = true)
    with torch.no_grad():  # no gradients needed
        for x, y, rel in groups:  # each group
            pred = model(x, rel).argmax(1)  # predicted classes
            cm += torch.bincount(y * NUM_CLASSES + pred, minlength=NUM_CLASSES**2).reshape(NUM_CLASSES, NUM_CLASSES)  # add counts
    tp = cm.diag().float()  # correct per class
    fp = cm.sum(0).float() - tp  # false alarms per class
    fn = cm.sum(1).float() - tp  # misses per class
    iou = tp / (tp + fp + fn).clamp(min=1)  # per-class IoU
    f1 = 2 * tp / (2 * tp + fp + fn).clamp(min=1)  # per-class F1
    return (tp.sum() / cm.sum()).item(), f1.mean().item(), iou, f1, cm  # results

start_epoch, history = 0, []  # fresh start by default
if os.path.exists(CKPT):  # resume if a checkpoint exists
    ck = torch.load(CKPT, weights_only=False)  # load it
    model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"])  # restore weights and optimizer
    start_epoch, history = ck["epoch"], ck["history"]  # restore progress
    print("Resuming from epoch", start_epoch)  # tell the user

for epoch in range(start_epoch, EPOCHS):  # training loop
    t0 = time.time()  # start timer
    model.train()  # training mode
    total, nfaces = 0.0, 0  # running loss and face count
    for gi in torch.randperm(len(gtr)).tolist():  # groups in random order
        x, y, rel = gtr[gi]  # one group
        opt.zero_grad()  # clear gradients
        loss = loss_fn(model(x, rel), y)  # forward pass and loss
        loss.backward()  # gradients
        opt.step()  # update weights
        total += loss.item() * len(y); nfaces += len(y)  # accumulate loss
    acc, mf1, _, _, _ = evaluate(gva)  # validation check
    history.append({"epoch": epoch + 1, "train_loss": total / nfaces, "val_acc": acc, "val_macro_f1": mf1})  # log
    print(f"epoch {epoch+1}/{EPOCHS} loss {total/nfaces:.4f} val_acc {acc:.4f} val_macroF1 {mf1:.4f} ({time.time()-t0:.0f}s)")  # progress
    torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "epoch": epoch + 1, "history": history}, CKPT + ".tmp")  # temp save
    os.replace(CKPT + ".tmp", CKPT)  # safe replace

acc, mf1, iou, f1, cm = evaluate(gva)  # final evaluation
print(f"\nFINAL val accuracy {acc:.4f} | macro-F1 {mf1:.4f} | mean IoU {iou.mean():.4f}")  # summary
print("per-class IoU:", [round(v, 3) for v in iou.tolist()])  # IoU for classes 0-24
if not TEST_RUN:  # only save official results for the full run
    helpers.save_json({"val_accuracy": acc, "val_macro_f1": mf1, "val_mean_iou": iou.mean().item(),
                       "per_class_iou": iou.tolist(), "per_class_f1": f1.tolist(),
                       "confusion_matrix": cm.tolist(), "history": history}, "results", "chunk5_gnn_val.json")  # save to Drive
    helpers.mark_done("chunk5_gnn", "neighbour-aware GNN trained and evaluated on val")  # record progress
else:
    print("Quick test finished. Check the time per epoch, then set TEST_RUN = False.")  # next step hint
TEST_RUN = False  # False = full run on all data (15 epochs, saves results to Drive)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
train groups: 185 | val groups: 39
epoch 1/30 loss 1.1793 val_acc 0.6780 val_macroF1 0.5605 (119s)
epoch 2/30 loss 0.7824 val_acc 0.7245 val_macroF1 0.6642 (117s)
epoch 3/30 loss 0.6446 val_acc 0.7718 val_macroF1 0.7290 (117s)
epoch 4/30 loss 0.5768 val_acc 0.7788 val_macroF1 0.7500 (119s)
epoch 5/30 loss 0.5266 val_acc 0.8137 val_macroF1 0.7853 (121s)
epoch 6/30 loss 0.4897 val_acc 0.8236 val_macroF1 0.7981 (119s)
epoch 7/30 loss 0.4541 val_acc 0.8403 val_macroF1 0.8158 (120s)
epoch 8/30 loss 0.4175 val_acc 0.8480 val_macroF1 0.8286 (118s)
epoch 9/30 loss 0.3834 val_acc 0.8601 val_macroF1 0.8420 (119s)
epoch 10/30 loss 0.3609 val_acc 0.8719 val_macroF1 0.8511 (120s)
epoch 11/30 loss 0.3309 val_acc 0.8768 val_macroF1 0.8550 (118s)
epoch 12/30 loss 0.3088 val_acc 0.8858 val_macroF1 0.8630 (120s)
epoch 13/30 loss 0.2926 val_acc 0.8903 val_macroF1 0.8660 (120s)


In [12]:
#chunk 5
#3 ensuring the 2 runs are saved in different files
import sys  # lets us add folders to Python's search path
from google.colab import drive  # Colab Drive helper
drive.mount('/content/drive')  # connect Drive (safe to run twice)
sys.path.append("/content/drive/MyDrive/mfcad_project")  # make helpers.py importable
import helpers  # our saving/progress helpers

v2 = helpers.load_json("results", "chunk5_gnn_val.json")  # this file currently holds the NEW (v2) results
helpers.save_json(v2, "results", "chunk5b_gnn_val.json")  # save them under the correct name
print("v2 saved as chunk5b. accuracy:", round(v2["val_accuracy"], 4))  # should print 0.9291

# Rewrite the v1 file with the numbers we saw in the earlier output (partial: no confusion matrix)
v1 = {"note": "PARTIAL v1 results (hidden 64, 4 layers, 15 epochs), re-entered by hand from the printed output",
      "val_accuracy": 0.8012, "val_macro_f1": 0.7754, "val_mean_iou": 0.6635,
      "per_class_iou": [0.543, 0.976, 0.461, 0.504, 0.694, 0.549, 0.362, 0.853, 0.402, 0.572,
                        0.252, 0.952, 0.909, 0.429, 0.456, 0.573, 0.967, 0.52, 0.826, 0.881,
                        0.583, 0.926, 0.516, 0.901, 0.986]}  # per-class IoU for classes 0-24
helpers.save_json(v1, "results", "chunk5_gnn_val.json")  # replaces the v2 content with the v1 summary
helpers.mark_done("chunk5b_gnn", "bigger GNN (hidden 128, 6 layers, 30 epochs) trained and evaluated on val")  # record progress

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Saved: /content/drive/MyDrive/mfcad_project/results/chunk5b_gnn_val.json
v2 saved as chunk5b. accuracy: 0.9291
Saved: /content/drive/MyDrive/mfcad_project/results/chunk5_gnn_val.json
Marked done: chunk5b_gnn


In [13]:
#chunk 6
import os  # file paths and checks
import sys  # lets us add folders to Python's search path
from google.colab import drive  # Colab Drive helper
drive.mount('/content/drive')  # connect Drive (safe to run twice)
PROJECT = "/content/drive/MyDrive/mfcad_project"  # project folder
sys.path.append(PROJECT)  # make helpers.py importable

ZIP = f"{PROJECT}/data/MFCAD_dataset.zip"  # the dataset zip on Drive
H5DIR = "/content/mfcad/MFCAD++_dataset/hierarchical_graphs"  # where the h5 files should end up

# Unzip ONLY the h5 graph files (the "*hierarchical_graphs*" pattern matches any folder prefix)
if not os.path.exists(f"{H5DIR}/val_MFCAD++.h5"):  # skip if already unzipped
    !mkdir -p /content/mfcad  # make the target folder
    !unzip -q -o "{ZIP}" "*hierarchical_graphs*" -d /content/mfcad  # extract just the h5 files
print("h5 folder content:", sorted(os.listdir(H5DIR)) if os.path.isdir(H5DIR) else "NOT FOUND")  # check the result

import h5py  # to read the h5 file
with h5py.File(f"{H5DIR}/val_MFCAD++.h5", "r") as f:  # open the val file
    names = sorted(f.keys(), key=int)  # batch names, sorted as numbers
    print("number of batches:", len(names))  # should be 312
    g = f[names[0]]  # first batch ('0')
    print("\nStructure of batch", names[0], "(name | shape | dtype):")  # section title
    def show(name, obj):  # called for every dataset or group inside the batch
        if isinstance(obj, h5py.Dataset):  # a real array
            print(f"  {name:28s} {str(obj.shape):18s} {obj.dtype}")  # print its shape and type
        else:  # a group that contains more arrays
            print(f"  {name:28s} (group)")  # just mark it as a group
    g.visititems(show)  # walk through everything in the batch
    print("\nfirst 3 rows of V_2:\n", g["V_2"][:3])  # mesh facet features
    print("idx (first 3 rows):\n", g["idx"][:3])  # part boundaries

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
h5 folder content: ['test_MFCAD++.h5', 'training_MFCAD++.h5', 'val_MFCAD++.h5']
number of batches: 312

Structure of batch 0 (name | shape | dtype):
  A_1_idx                      (4567, 2)          int32
  A_1_shape                    (2,)               int32
  A_1_values                   (4567,)            float32
  A_2_idx                      (26908, 2)         int32
  A_2_shape                    (2,)               int32
  A_2_values                   (26908,)           float32
  A_3_idx                      (8971, 2)          int32
  A_3_shape                    (2,)               int32
  A_3_values                   (8971,)            float32
  A_4_idx                      (8971, 2)          int32
  A_4_shape                    (2,)               int32
  A_4_values                   (8971,)            float32
  CAD_model                    (29,)      

In [14]:
#chunk 6
#2
from google.colab import drive  # Colab Drive helper
drive.mount('/content/drive')  # connect Drive (safe to run twice)
import h5py  # to read the h5 file
import numpy as np  # array maths

H5 = "/content/mfcad/MFCAD++_dataset/hierarchical_graphs/val_MFCAD++.h5"  # val file
with h5py.File(H5, "r") as f:  # open it
    g = f["0"]  # first batch
    for k in ["A_1_shape", "A_2_shape", "A_3_shape", "A_4_shape"]:  # sparse matrix shapes
        print(k, g[k][:])  # print each
    a3 = g["A_3_idx"][:]  # face<->facet pairs
    print("A_3 first rows:", a3[:4].tolist(), "| values:", np.unique(g["A_3_values"][:]))  # sample rows and values
    print("A_3 max per column:", a3.max(0))  # the column with the smaller max should be faces (0-874)
    n_faces, n_facets = g["V_1"].shape[0], g["V_2"].shape[0]  # counts in this batch
    idx = g["idx"][:]  # part boundaries
    print("idx last row:", idx[-1], "| faces:", n_faces, "| facets:", n_facets)  # compare with counts
    print("V_2 min per column:", g["V_2"][:].min(0), "| max:", g["V_2"][:].max(0))  # value ranges
    print("A_2 first rows:", g["A_2_idx"][:4].tolist(), "| max:", g["A_2_idx"][:].max())  # facet adjacency sample
    fcol = int(a3.max(0).argmin())  # column that looks like the face index
    tcol = 1 - fcol  # the other column should be facets
    print("face column:", fcol, "| every facet appears once:", len(np.unique(a3[:, tcol])) == n_facets)  # one face per facet?
    facet_face = np.zeros(n_facets, dtype=int)  # face number of each facet
    facet_face[a3[:, tcol]] = a3[:, fcol]  # fill from A_3
    print("facet_labels == labels of its face:", bool(np.all(g["labels"][:][facet_face] == g["facet_labels"][:])))  # label consistency

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
A_1_shape [875 875]
A_2_shape [8971 8971]
A_3_shape [8971  875]
A_4_shape [ 875 8971]
A_3 first rows: [[0, 0], [1, 0], [2, 0], [3, 0]] | values: [1.]
A_3 max per column: [8970  874]
idx last row: [ 874 8970] | faces: 875 | facets: 8971
V_2 min per column: [    0.         0.         0.     -4397.2676] | max: [1.0000000e+00 1.0000000e+00 1.0000000e+00 3.9519426e+03]
A_2 first rows: [[0, 3], [0, 6], [0, 104], [1, 2]] | max: 8970
face column: 1 | every facet appears once: True
facet_labels == labels of its face: True


In [15]:
#chunk 6a
#1
import os  # file paths and safe replacing
import sys  # lets us add folders to Python's search path
import time  # to time each split
import h5py  # to read the h5 files
import numpy as np  # array maths
import torch  # to load and save the caches

from google.colab import drive  # Colab Drive helper
drive.mount('/content/drive')  # connect Drive (safe to run twice)
PROJECT = "/content/drive/MyDrive/mfcad_project"  # project folder
sys.path.append(PROJECT)  # make helpers.py importable
import helpers  # our saving/progress helpers

H5DIR = "/content/mfcad/MFCAD++_dataset/hierarchical_graphs"  # folder with the h5 files
assert os.path.isdir(H5DIR), "h5 folder missing: re-run the partial unzip from chunk 6.0"  # stop early if the unzip is gone

jobs = [("val", "val_MFCAD++.h5"), ("test", "test_MFCAD++.h5"), ("train", "training_MFCAD++.h5")]  # (split, h5 file)

def build_mesh(split, h5_name):  # build the mesh-level cache for one split
    face_cache = torch.load(f"{PROJECT}/data/cache_{split}.pt", weights_only=False)  # our face cache, for cross-checks
    out, stats = [], {"n_batches": 0, "n_facets": 0, "n_edges": 0, "a2_asymmetric_batches": 0,  # counters for the summary
                      "a2_self_loops": 0, "idx_facet_mismatch_batches": 0, "idx_facet_bad_batches": 0}
    with h5py.File(f"{H5DIR}/{h5_name}", "r") as f:  # open the h5 file
        for k, fb in enumerate(face_cache["batches"]):  # go through the face-cache batches in order
            g = f[str(fb["name"])]  # the same batch inside the h5 file
            n_faces, n_facets = len(fb["y"]), g["V_2"].shape[0]  # faces and facets in this batch
            assert g["V_1"].shape[0] == n_faces, "face count differs from the face cache"  # same faces as the cache
            a3 = g["A_3_idx"][:].astype(np.int64)  # (facet, face) pairs
            assert len(a3) == n_facets and len(np.unique(a3[:, 0])) == n_facets, "A_3 must list each facet once"  # one face per facet
            facet_face = np.empty(n_facets, dtype=np.int64)  # face number of every facet
            facet_face[a3[:, 0]] = a3[:, 1]  # fill from A_3 (column 0 = facet, column 1 = face)
            assert facet_face.max() < n_faces, "a facet points to a face that does not exist"  # valid face numbers
            y = np.asarray(fb["y"]).astype(np.int64)  # face labels from our cache
            assert np.array_equal(g["facet_labels"][:].astype(np.int64), y[facet_face]), "facet labels differ from face labels"  # labels agree
            v2 = g["V_2"][:].astype(np.float32)  # raw facet features (facets,4)
            d = v2[:, 3]  # the unscaled 4th column
            v2[:, 3] = np.sign(d) * np.log1p(np.abs(d))  # signed log keeps huge values from dominating
            a2 = g["A_2_idx"][:].astype(np.int64)  # facet adjacency pairs
            keys = a2[:, 0] * n_facets + a2[:, 1]  # one number per directed edge
            rev = a2[:, 1] * n_facets + a2[:, 0]  # the reverse of every edge
            if not np.all(np.isin(rev, keys)):  # is some edge missing its reverse?
                stats["a2_asymmetric_batches"] += 1  # count it
            stats["a2_self_loops"] += int((a2[:, 0] == a2[:, 1]).sum())  # count edges from a facet to itself
            # Check the guess that idx[:,1] is the inclusive last facet of each model (global counter)
            idx = g["idx"][:].astype(np.int64)  # part boundaries (faces, facets)
            sizes = np.diff(np.concatenate([[idx[-1, 1] - n_facets], idx[:, 1]]))  # facets per model
            if sizes.min() <= 0 or sizes.sum() != n_facets:  # sizes impossible?
                stats["idx_facet_bad_batches"] += 1  # hypothesis fails here
            else:  # sizes are possible, so compare with the face-based part numbers
                fm_facet = np.repeat(np.arange(len(sizes)), sizes)  # part number per facet from idx[:,1]
                fm_face = fb["face_model"].numpy().astype(np.int64)  # part number per face (correct, from idx[:,0])
                if not np.array_equal(fm_facet, fm_face[facet_face]):  # do both views agree?
                    stats["idx_facet_mismatch_batches"] += 1  # count disagreements
            out.append({"name": fb["name"],  # keep the batch name
                        "v2": torch.from_numpy(v2),  # (facets,4) float32
                        "a2": torch.from_numpy(a2.astype(np.int32).T.copy()),  # (2,E) int32 facet edges
                        "facet_face": torch.from_numpy(facet_face.astype(np.int32))})  # (facets,) face of each facet
            stats["n_batches"] += 1; stats["n_facets"] += n_facets; stats["n_edges"] += len(a2)  # update totals
            if (k + 1) % 200 == 0:  # progress line
                print(f"  {split}: processed {k+1} / {len(face_cache['batches'])} batches")  # show progress
    path = f"{PROJECT}/data/mesh_cache_{split}.pt"  # output file
    torch.save({"batches": out, "summary": stats}, path + ".tmp")  # save to a temp file first
    os.replace(path + ".tmp", path)  # then replace safely
    helpers.save_json(stats, "notes", f"chunk6_mesh_summary_{split}.json")  # keep the summary on Drive
    print("Saved:", path); print(stats)  # report

for split, h5_name in jobs:  # val first (small), then test, then train
    if os.path.exists(f"{PROJECT}/data/mesh_cache_{split}.pt"):  # already built?
        print("skip (already exists):", split); continue  # do not rebuild
    t0 = time.time()  # start timer
    print("Building mesh cache:", split)  # announce
    build_mesh(split, h5_name)  # do the work
    print(f"  took {time.time()-t0:.0f}s")  # report the time

helpers.mark_done("chunk6a_mesh_cache", "mesh caches built and cross-checked against face caches")  # record progress

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Building mesh cache: val
  val: processed 200 / 312 batches
Saved: /content/drive/MyDrive/mfcad_project/notes/chunk6_mesh_summary_val.json
Saved: /content/drive/MyDrive/mfcad_project/data/mesh_cache_val.pt
{'n_batches': 312, 'n_facets': 2733599, 'n_edges': 8199930, 'a2_asymmetric_batches': 0, 'a2_self_loops': 0, 'idx_facet_mismatch_batches': 0, 'idx_facet_bad_batches': 0}
  took 9s
Building mesh cache: test
  test: processed 200 / 317 batches
Saved: /content/drive/MyDrive/mfcad_project/notes/chunk6_mesh_summary_test.json
Saved: /content/drive/MyDrive/mfcad_project/data/mesh_cache_test.pt
{'n_batches': 317, 'n_facets': 2757805, 'n_edges': 8272088, 'a2_asymmetric_batches': 0, 'a2_self_loops': 0, 'idx_facet_mismatch_batches': 0, 'idx_facet_bad_batches': 0}
  took 15s
Building mesh cache: train
  train: processed 200 / 1473 batches
  train: processed 400 / 1473 b

In [16]:
#chunk 6b
#1 (2 epoch test run)

import os  # file paths and safe replacing
import sys  # lets us add folders to Python's search path
import time  # to time each epoch
import torch  # main library
import torch.nn as nn  # network building blocks

from google.colab import drive  # Colab Drive helper
drive.mount('/content/drive')  # connect Drive (safe to run twice)
PROJECT = "/content/drive/MyDrive/mfcad_project"  # project folder
sys.path.append(PROJECT)  # make helpers.py importable
import helpers  # our saving/progress helpers

TEST_RUN = True  # True = quick check on a subset; set to False for the full run
torch.manual_seed(0)  # fixed seed for repeatable results
NUM_CLASSES = 25  # labels 0-24
HIDDEN, LAYERS = 128, 6  # face-level size and rounds (same as GNN v2)
FHID, FLAYERS = 32, 2  # facet-level size and rounds (small, because there are ~10x more facets)
GROUP = 8  # h5 batches merged into one training step
LR = 2e-3  # learning rate
EPOCHS = 2 if TEST_RUN else 30  # passes over the training data
CKPT = f"{PROJECT}/models/{'hier_test_ckpt.pt' if TEST_RUN else 'hier_ckpt.pt'}"  # checkpoint file (new names)

def build_groups(fbs, mbs):  # merge several batches into groups; fbs = face batches, mbs = mesh batches
    groups = []  # result list
    for s in range(0, len(fbs), GROUP):  # step through batches in blocks
        foff, toff = 0, 0  # running offsets for faces and facets
        xs, ys, eis, ets, v2s, a2s, ffs = [], [], [], [], [], [], []  # collectors
        for fb, mb in zip(fbs[s:s + GROUP], mbs[s:s + GROUP]):  # each batch in this block
            assert fb["name"] == mb["name"], "face and mesh caches are out of step"  # same batch in both caches
            n, m = len(fb["y"]), len(mb["v2"])  # faces and facets in this batch
            xs.append(torch.as_tensor(fb["x"]).float())  # face inputs (n,16)
            ys.append(torch.as_tensor(fb["y"]).long())  # face classes
            eis.append(torch.as_tensor(fb["edge_index"]).long() + foff)  # face edges, shifted to group numbering
            ets.append(torch.as_tensor(fb["edge_type"]).long())  # face edge types
            v2s.append(mb["v2"].float())  # facet features (m,4)
            a2s.append(mb["a2"].long() + toff)  # facet edges, shifted to group numbering
            ffs.append(mb["facet_face"].long() + foff)  # which face each facet belongs to, shifted
            foff += n; toff += m  # next batch starts after these
        x, y = torch.cat(xs), torch.cat(ys)  # all faces of the group
        ei, et = torch.cat(eis, dim=1), torch.cat(ets)  # all face edges of the group
        rel = []  # face edges split by type
        for t in range(3):  # convex, concave, smooth
            msk = et == t  # edges of this type
            src, dst = ei[0][msk], ei[1][msk]  # sender and receiver faces
            deg = torch.bincount(dst, minlength=foff).float().clamp(min=1)  # neighbours per face
            rel.append((src, dst, 1.0 / deg))  # store with 1/degree
        a2 = torch.cat(a2s, dim=1)  # all facet edges of the group
        fdeg = torch.bincount(a2[1], minlength=toff).float().clamp(min=1)  # neighbours per facet
        ff = torch.cat(ffs)  # face of every facet
        cnt = torch.bincount(ff, minlength=foff).float().clamp(min=1)  # facets per face (for the mean)
        groups.append({"x": x, "y": y, "rel": rel, "v2": torch.cat(v2s),  # keep everything for this group
                       "frel": (a2[0], a2[1], 1.0 / fdeg), "ff": ff, "cnt": cnt})
    return groups  # done

def load_groups(split, limit):  # load face + mesh cache of a split and build groups
    fc = torch.load(f"{PROJECT}/data/cache_{split}.pt", weights_only=False)["batches"]  # face cache
    mc = torch.load(f"{PROJECT}/data/mesh_cache_{split}.pt", weights_only=False)["batches"]  # mesh cache
    if limit:  # optional subset for the quick test
        fc, mc = fc[:limit], mc[:limit]  # same batches from both
    return build_groups(fc, mc)  # merged groups

gtr = load_groups("train", 100 if TEST_RUN else None)  # training groups
gva = load_groups("val", 30 if TEST_RUN else None)  # validation groups
print("train groups:", len(gtr), "| val groups:", len(gva))  # 13/4 = quick test, 185/39 = full run

class FacetLayer(nn.Module):  # one round of "look at neighbouring mesh facets"
    def __init__(self, d):  # d = size of facet representation
        super().__init__()  # standard setup
        self.self_lin = nn.Linear(d, d)  # facet's own information
        self.nbr_lin = nn.Linear(d, d, bias=False)  # information from neighbouring facets
    def forward(self, h, rel):  # h = (facets, d)
        src, dst, inv = rel  # edge lists and 1/degree
        agg = torch.zeros_like(h).index_add_(0, dst, h[src]) * inv.unsqueeze(1)  # mean of neighbours
        return h + torch.relu(self.self_lin(h) + self.nbr_lin(agg))  # residual update

class Layer(nn.Module):  # one round of "look at neighbouring faces" (same as GNN v2)
    def __init__(self, d):  # d = size of face representation
        super().__init__()  # standard setup
        self.self_lin = nn.Linear(d, d)  # the face's own information
        self.nbr_lin = nn.ModuleList([nn.Linear(d, d, bias=False) for _ in range(3)])  # one path per edge type
    def forward(self, h, rel):  # h = (faces, d)
        out = self.self_lin(h)  # start with the face itself
        for t, (src, dst, invdeg) in enumerate(rel):  # each edge type
            agg = torch.zeros_like(h).index_add_(0, dst, h[src]) * invdeg.unsqueeze(1)  # mean of neighbours
            out = out + self.nbr_lin[t](agg)  # add neighbour information
        return h + torch.relu(out)  # residual connection

class Net(nn.Module):  # facets -> faces -> classes
    def __init__(self):  # build the parts
        super().__init__()  # standard setup
        self.f_inp = nn.Linear(4, FHID)  # 4 facet features -> facet size
        self.f_layers = nn.ModuleList([FacetLayer(FHID) for _ in range(FLAYERS)])  # facet rounds
        self.pool_lin = nn.Linear(FHID, HIDDEN)  # pooled facet information -> face size
        self.inp = nn.Linear(16, HIDDEN)  # 16 face features -> face size
        self.layers = nn.ModuleList([Layer(HIDDEN) for _ in range(LAYERS)])  # face rounds
        self.out = nn.Linear(HIDDEN, NUM_CLASSES)  # face size -> 25 class scores
    def forward(self, g):  # g = one group dictionary
        f = torch.relu(self.f_inp(g["v2"]))  # embed facet features
        for layer in self.f_layers:  # facet rounds
            f = layer(f, g["frel"])  # mix in neighbouring facets
        pooled = torch.zeros(len(g["x"]), FHID).index_add_(0, g["ff"], f) / g["cnt"].unsqueeze(1)  # mean of each face's facets
        h = torch.relu(self.inp(g["x"]) + self.pool_lin(pooled))  # combine face features with facet summary
        for layer in self.layers:  # face rounds
            h = layer(h, g["rel"])  # mix in neighbouring faces
        return self.out(h)  # class scores per face

model = Net()  # create the model
opt = torch.optim.Adam(model.parameters(), lr=LR)  # optimizer
loss_fn = nn.CrossEntropyLoss()  # classification loss

def evaluate(groups):  # accuracy, macro-F1, per-class IoU
    model.eval()  # evaluation mode
    cm = torch.zeros(NUM_CLASSES, NUM_CLASSES, dtype=torch.long)  # confusion matrix (rows = true)
    with torch.no_grad():  # no gradients needed
        for g in groups:  # each group
            pred = model(g).argmax(1)  # predicted classes
            cm += torch.bincount(g["y"] * NUM_CLASSES + pred, minlength=NUM_CLASSES**2).reshape(NUM_CLASSES, NUM_CLASSES)  # add counts
    tp = cm.diag().float()  # correct per class
    fp = cm.sum(0).float() - tp  # false alarms
    fn = cm.sum(1).float() - tp  # misses
    iou = tp / (tp + fp + fn).clamp(min=1)  # per-class IoU
    f1 = 2 * tp / (2 * tp + fp + fn).clamp(min=1)  # per-class F1
    return (tp.sum() / cm.sum()).item(), f1.mean().item(), iou, f1, cm  # results

start_epoch, history = 0, []  # fresh start by default
if os.path.exists(CKPT):  # resume if a checkpoint exists
    ck = torch.load(CKPT, weights_only=False)  # load it
    model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"])  # restore weights and optimizer
    start_epoch, history = ck["epoch"], ck["history"]  # restore progress
    print("Resuming from epoch", start_epoch)  # tell the user

for epoch in range(start_epoch, EPOCHS):  # training loop
    t0 = time.time()  # start timer
    model.train()  # training mode
    total, nfaces = 0.0, 0  # running loss and face count
    for gi in torch.randperm(len(gtr)).tolist():  # groups in random order
        g = gtr[gi]  # one group
        opt.zero_grad()  # clear gradients
        loss = loss_fn(model(g), g["y"])  # forward pass and loss
        loss.backward()  # gradients
        opt.step()  # update weights
        total += loss.item() * len(g["y"]); nfaces += len(g["y"])  # accumulate loss
    acc, mf1, _, _, _ = evaluate(gva)  # validation check
    history.append({"epoch": epoch + 1, "train_loss": total / nfaces, "val_acc": acc, "val_macro_f1": mf1})  # log
    print(f"epoch {epoch+1}/{EPOCHS} loss {total/nfaces:.4f} val_acc {acc:.4f} val_macroF1 {mf1:.4f} ({time.time()-t0:.0f}s)")  # progress
    torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "epoch": epoch + 1, "history": history}, CKPT + ".tmp")  # temp save
    os.replace(CKPT + ".tmp", CKPT)  # safe replace

acc, mf1, iou, f1, cm = evaluate(gva)  # final evaluation
print(f"\nFINAL val accuracy {acc:.4f} | macro-F1 {mf1:.4f} | mean IoU {iou.mean():.4f}")  # summary
print("per-class IoU:", [round(v, 3) for v in iou.tolist()])  # IoU for classes 0-24
if not TEST_RUN:  # only save official results for the full run
    helpers.save_json({"val_accuracy": acc, "val_macro_f1": mf1, "val_mean_iou": iou.mean().item(),
                       "per_class_iou": iou.tolist(), "per_class_f1": f1.tolist(),
                       "confusion_matrix": cm.tolist(), "history": history}, "results", "chunk6_hier_val.json")  # save to Drive
    helpers.mark_done("chunk6_hier", "simplified hierarchical (facet+face) model trained and evaluated on val")  # record progress
else:
    print("Quick test finished. Multiply the epoch time by about 15 to estimate a full epoch.")  # hint

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
train groups: 13 | val groups: 4
epoch 1/2 loss 2.8775 val_acc 0.3740 val_macroF1 0.1298 (13s)
epoch 2/2 loss 2.0151 val_acc 0.4449 val_macroF1 0.1851 (15s)

FINAL val accuracy 0.4449 | macro-F1 0.1851 | mean IoU 0.1406
per-class IoU: [0.0, 0.848, 0.0, 0.0, 0.216, 0.0, 0.0, 0.0, 0.001, 0.019, 0.003, 0.448, 0.0, 0.128, 0.045, 0.318, 0.275, 0.0, 0.0, 0.0, 0.122, 0.0, 0.046, 0.102, 0.943]
Quick test finished. Multiply the epoch time by about 15 to estimate a full epoch.


In [18]:
#6 c
# hierarchial 1

import os  # file paths and safe replacing
import sys  # lets us add folders to Python's search path
import time  # to time each epoch
import torch  # main library
import torch.nn as nn  # network building blocks

from google.colab import drive  # Colab Drive helper
drive.mount('/content/drive')  # connect Drive (safe to run twice)
PROJECT = "/content/drive/MyDrive/mfcad_project"  # project folder
sys.path.append(PROJECT)  # make helpers.py importable
import helpers  # our saving/progress helpers

TEST_RUN = False # True = quick check on a subset; set to False for the full run
torch.manual_seed(0)  # fixed seed for repeatable results
NUM_CLASSES = 25  # labels 0-24
HIDDEN, LAYERS = 128, 6  # face-level size and rounds (same as GNN v2)
FHID, FLAYERS = 32, 2  # facet-level size and rounds (small, because there are ~10x more facets)
GROUP = 8  # h5 batches merged into one training step
LR = 2e-3  # learning rate
EPOCHS = 2 if TEST_RUN else 30  # passes over the training data
CKPT = f"{PROJECT}/models/{'hier_test_ckpt.pt' if TEST_RUN else 'hier_ckpt.pt'}"  # checkpoint file (new names)

def build_groups(fbs, mbs):  # merge several batches into groups; fbs = face batches, mbs = mesh batches
    groups = []  # result list
    for s in range(0, len(fbs), GROUP):  # step through batches in blocks
        foff, toff = 0, 0  # running offsets for faces and facets
        xs, ys, eis, ets, v2s, a2s, ffs = [], [], [], [], [], [], []  # collectors
        for fb, mb in zip(fbs[s:s + GROUP], mbs[s:s + GROUP]):  # each batch in this block
            assert fb["name"] == mb["name"], "face and mesh caches are out of step"  # same batch in both caches
            n, m = len(fb["y"]), len(mb["v2"])  # faces and facets in this batch
            xs.append(torch.as_tensor(fb["x"]).float())  # face inputs (n,16)
            ys.append(torch.as_tensor(fb["y"]).long())  # face classes
            eis.append(torch.as_tensor(fb["edge_index"]).long() + foff)  # face edges, shifted to group numbering
            ets.append(torch.as_tensor(fb["edge_type"]).long())  # face edge types
            v2s.append(mb["v2"].float())  # facet features (m,4)
            a2s.append(mb["a2"].long() + toff)  # facet edges, shifted to group numbering
            ffs.append(mb["facet_face"].long() + foff)  # which face each facet belongs to, shifted
            foff += n; toff += m  # next batch starts after these
        x, y = torch.cat(xs), torch.cat(ys)  # all faces of the group
        ei, et = torch.cat(eis, dim=1), torch.cat(ets)  # all face edges of the group
        rel = []  # face edges split by type
        for t in range(3):  # convex, concave, smooth
            msk = et == t  # edges of this type
            src, dst = ei[0][msk], ei[1][msk]  # sender and receiver faces
            deg = torch.bincount(dst, minlength=foff).float().clamp(min=1)  # neighbours per face
            rel.append((src, dst, 1.0 / deg))  # store with 1/degree
        a2 = torch.cat(a2s, dim=1)  # all facet edges of the group
        fdeg = torch.bincount(a2[1], minlength=toff).float().clamp(min=1)  # neighbours per facet
        ff = torch.cat(ffs)  # face of every facet
        cnt = torch.bincount(ff, minlength=foff).float().clamp(min=1)  # facets per face (for the mean)
        groups.append({"x": x, "y": y, "rel": rel, "v2": torch.cat(v2s),  # keep everything for this group
                       "frel": (a2[0], a2[1], 1.0 / fdeg), "ff": ff, "cnt": cnt})
    return groups  # done

def load_groups(split, limit):  # load face + mesh cache of a split and build groups
    fc = torch.load(f"{PROJECT}/data/cache_{split}.pt", weights_only=False)["batches"]  # face cache
    mc = torch.load(f"{PROJECT}/data/mesh_cache_{split}.pt", weights_only=False)["batches"]  # mesh cache
    if limit:  # optional subset for the quick test
        fc, mc = fc[:limit], mc[:limit]  # same batches from both
    return build_groups(fc, mc)  # merged groups

gtr = load_groups("train", 100 if TEST_RUN else None)  # training groups
gva = load_groups("val", 30 if TEST_RUN else None)  # validation groups
print("train groups:", len(gtr), "| val groups:", len(gva))  # 13/4 = quick test, 185/39 = full run

class FacetLayer(nn.Module):  # one round of "look at neighbouring mesh facets"
    def __init__(self, d):  # d = size of facet representation
        super().__init__()  # standard setup
        self.self_lin = nn.Linear(d, d)  # facet's own information
        self.nbr_lin = nn.Linear(d, d, bias=False)  # information from neighbouring facets
    def forward(self, h, rel):  # h = (facets, d)
        src, dst, inv = rel  # edge lists and 1/degree
        agg = torch.zeros_like(h).index_add_(0, dst, h[src]) * inv.unsqueeze(1)  # mean of neighbours
        return h + torch.relu(self.self_lin(h) + self.nbr_lin(agg))  # residual update

class Layer(nn.Module):  # one round of "look at neighbouring faces" (same as GNN v2)
    def __init__(self, d):  # d = size of face representation
        super().__init__()  # standard setup
        self.self_lin = nn.Linear(d, d)  # the face's own information
        self.nbr_lin = nn.ModuleList([nn.Linear(d, d, bias=False) for _ in range(3)])  # one path per edge type
    def forward(self, h, rel):  # h = (faces, d)
        out = self.self_lin(h)  # start with the face itself
        for t, (src, dst, invdeg) in enumerate(rel):  # each edge type
            agg = torch.zeros_like(h).index_add_(0, dst, h[src]) * invdeg.unsqueeze(1)  # mean of neighbours
            out = out + self.nbr_lin[t](agg)  # add neighbour information
        return h + torch.relu(out)  # residual connection

class Net(nn.Module):  # facets -> faces -> classes
    def __init__(self):  # build the parts
        super().__init__()  # standard setup
        self.f_inp = nn.Linear(4, FHID)  # 4 facet features -> facet size
        self.f_layers = nn.ModuleList([FacetLayer(FHID) for _ in range(FLAYERS)])  # facet rounds
        self.pool_lin = nn.Linear(FHID, HIDDEN)  # pooled facet information -> face size
        self.inp = nn.Linear(16, HIDDEN)  # 16 face features -> face size
        self.layers = nn.ModuleList([Layer(HIDDEN) for _ in range(LAYERS)])  # face rounds
        self.out = nn.Linear(HIDDEN, NUM_CLASSES)  # face size -> 25 class scores
    def forward(self, g):  # g = one group dictionary
        f = torch.relu(self.f_inp(g["v2"]))  # embed facet features
        for layer in self.f_layers:  # facet rounds
            f = layer(f, g["frel"])  # mix in neighbouring facets
        pooled = torch.zeros(len(g["x"]), FHID).index_add_(0, g["ff"], f) / g["cnt"].unsqueeze(1)  # mean of each face's facets
        h = torch.relu(self.inp(g["x"]) + self.pool_lin(pooled))  # combine face features with facet summary
        for layer in self.layers:  # face rounds
            h = layer(h, g["rel"])  # mix in neighbouring faces
        return self.out(h)  # class scores per face

model = Net()  # create the model
opt = torch.optim.Adam(model.parameters(), lr=LR)  # optimizer
loss_fn = nn.CrossEntropyLoss()  # classification loss

def evaluate(groups):  # accuracy, macro-F1, per-class IoU
    model.eval()  # evaluation mode
    cm = torch.zeros(NUM_CLASSES, NUM_CLASSES, dtype=torch.long)  # confusion matrix (rows = true)
    with torch.no_grad():  # no gradients needed
        for g in groups:  # each group
            pred = model(g).argmax(1)  # predicted classes
            cm += torch.bincount(g["y"] * NUM_CLASSES + pred, minlength=NUM_CLASSES**2).reshape(NUM_CLASSES, NUM_CLASSES)  # add counts
    tp = cm.diag().float()  # correct per class
    fp = cm.sum(0).float() - tp  # false alarms
    fn = cm.sum(1).float() - tp  # misses
    iou = tp / (tp + fp + fn).clamp(min=1)  # per-class IoU
    f1 = 2 * tp / (2 * tp + fp + fn).clamp(min=1)  # per-class F1
    return (tp.sum() / cm.sum()).item(), f1.mean().item(), iou, f1, cm  # results

start_epoch, history = 0, []  # fresh start by default
if os.path.exists(CKPT):  # resume if a checkpoint exists
    ck = torch.load(CKPT, weights_only=False)  # load it
    model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"])  # restore weights and optimizer
    start_epoch, history = ck["epoch"], ck["history"]  # restore progress
    print("Resuming from epoch", start_epoch)  # tell the user

for epoch in range(start_epoch, EPOCHS):  # training loop
    t0 = time.time()  # start timer
    model.train()  # training mode
    total, nfaces = 0.0, 0  # running loss and face count
    for gi in torch.randperm(len(gtr)).tolist():  # groups in random order
        g = gtr[gi]  # one group
        opt.zero_grad()  # clear gradients
        loss = loss_fn(model(g), g["y"])  # forward pass and loss
        loss.backward()  # gradients
        opt.step()  # update weights
        total += loss.item() * len(g["y"]); nfaces += len(g["y"])  # accumulate loss
    acc, mf1, _, _, _ = evaluate(gva)  # validation check
    history.append({"epoch": epoch + 1, "train_loss": total / nfaces, "val_acc": acc, "val_macro_f1": mf1})  # log
    print(f"epoch {epoch+1}/{EPOCHS} loss {total/nfaces:.4f} val_acc {acc:.4f} val_macroF1 {mf1:.4f} ({time.time()-t0:.0f}s)")  # progress
    torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "epoch": epoch + 1, "history": history}, CKPT + ".tmp")  # temp save
    os.replace(CKPT + ".tmp", CKPT)  # safe replace

acc, mf1, iou, f1, cm = evaluate(gva)  # final evaluation
print(f"\nFINAL val accuracy {acc:.4f} | macro-F1 {mf1:.4f} | mean IoU {iou.mean():.4f}")  # summary
print("per-class IoU:", [round(v, 3) for v in iou.tolist()])  # IoU for classes 0-24
if not TEST_RUN:  # only save official results for the full run
    helpers.save_json({"val_accuracy": acc, "val_macro_f1": mf1, "val_mean_iou": iou.mean().item(),
                       "per_class_iou": iou.tolist(), "per_class_f1": f1.tolist(),
                       "confusion_matrix": cm.tolist(), "history": history}, "results", "chunk6_hier_val.json")  # save to Drive
    helpers.mark_done("chunk6_hier", "simplified hierarchical (facet+face) model trained and evaluated on val")  # record progress
else:
    print("Quick test finished. Multiply the epoch time by about 15 to estimate a full epoch.")  # hint

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
train groups: 185 | val groups: 39
epoch 1/30 loss 1.1952 val_acc 0.7633 val_macroF1 0.6719 (183s)
epoch 2/30 loss 0.5585 val_acc 0.8383 val_macroF1 0.7676 (180s)
epoch 3/30 loss 0.4196 val_acc 0.8728 val_macroF1 0.8198 (187s)
epoch 4/30 loss 0.3465 val_acc 0.8878 val_macroF1 0.8360 (182s)
epoch 5/30 loss 0.2989 val_acc 0.8993 val_macroF1 0.8529 (182s)
epoch 6/30 loss 0.2704 val_acc 0.8991 val_macroF1 0.8571 (191s)
epoch 7/30 loss 0.2485 val_acc 0.9133 val_macroF1 0.8705 (185s)
epoch 8/30 loss 0.2281 val_acc 0.9173 val_macroF1 0.8793 (184s)
epoch 9/30 loss 0.2166 val_acc 0.9224 val_macroF1 0.8861 (183s)
epoch 10/30 loss 0.2025 val_acc 0.9205 val_macroF1 0.8814 (184s)
epoch 11/30 loss 0.1958 val_acc 0.9270 val_macroF1 0.8940 (183s)
epoch 12/30 loss 0.1851 val_acc 0.9333 val_macroF1 0.9033 (184s)
epoch 13/30 loss 0.1780 val_acc 0.9314 val_macroF1 0.9010 (184s)


In [19]:
#chunk 7
#checking where the confusions happened

import sys  # lets us add folders to Python's search path
from google.colab import drive  # Colab Drive helper
drive.mount('/content/drive')  # connect Drive (safe to run twice)
sys.path.append("/content/drive/MyDrive/mfcad_project")  # make helpers.py importable
import helpers  # our saving/progress helpers
import numpy as np  # array maths

# Class names 0-24 (from feature_labels.txt, as recorded in the handoff)
names = ["Chamfer", "Through hole", "Triangular passage", "Rectangular passage", "6-sided passage",
         "Triangular through slot", "Rectangular through slot", "Circular through slot",
         "Rectangular through step", "2-sided through step", "Slanted through step", "O-ring",
         "Blind hole", "Triangular pocket", "Rectangular pocket", "6-sided pocket",
         "Circular end pocket", "Rectangular blind slot", "Vertical circular end blind slot",
         "Horizontal circular end blind slot", "Triangular blind step", "Circular blind step",
         "Rectangular blind step", "Round", "Stock"]  # 25 class names

res = helpers.load_json("results", "chunk6_hier_val.json")  # results of the hierarchical model
cm = np.array(res["confusion_matrix"])  # confusion matrix, rows = true class, columns = predicted
print("matrix shape:", cm.shape, "| total faces:", int(cm.sum()))  # should be (25, 25) and 259490

# Per-class precision and recall (shows whether a class is missed or over-predicted)
tp = np.diag(cm).astype(float)  # correctly classified faces per class
recall = tp / np.maximum(cm.sum(1), 1)  # share of true faces of the class that were found
precision = tp / np.maximum(cm.sum(0), 1)  # share of predictions of the class that were right
print("\nWeakest 8 classes by IoU:")  # section title
for c in np.argsort(res["per_class_iou"])[:8]:  # the 8 lowest-IoU classes
    print(f"  {c:2d} {names[c]:36s} IoU {res['per_class_iou'][c]:.3f}  precision {precision[c]:.3f}  recall {recall[c]:.3f}  faces {int(cm[c].sum())}")  # one line per class

# The 12 most frequent mistakes (true class -> predicted class)
off = cm.copy()  # copy so the original stays unchanged
np.fill_diagonal(off, 0)  # ignore correct predictions
print("\nTop 12 confusions (true -> predicted : faces):")  # section title
for flat in np.argsort(off, axis=None)[::-1][:12]:  # largest off-diagonal counts first
    t, p = divmod(int(flat), 25)  # row = true class, column = predicted class
    print(f"  {names[t]} -> {names[p]} : {int(off[t, p])}  ({off[t, p] / cm[t].sum():.1%} of that class)")  # one mistake per line

helpers.mark_done("chunk7a_confusions", "confusion analysis of hierarchical model on val")  # record progress

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
matrix shape: (25, 25) | total faces: 259490

Weakest 8 classes by IoU:
  10 Slanted through step                 IoU 0.429  precision 0.720  recall 0.515  faces 6905
   8 Rectangular through step             IoU 0.556  precision 0.647  recall 0.799  faces 6954
   5 Triangular through slot              IoU 0.708  precision 0.830  recall 0.829  faces 2461
   0 Chamfer                              IoU 0.753  precision 0.861  recall 0.857  faces 3610
   6 Rectangular through slot             IoU 0.768  precision 0.862  recall 0.876  faces 4138
   2 Triangular passage                   IoU 0.843  precision 0.908  recall 0.922  faces 10744
   3 Rectangular passage                  IoU 0.846  precision 0.878  recall 0.958  faces 14742
  17 Rectangular blind slot               IoU 0.862  precision 0.948  recall 0.905  faces 3207

Top 12 confusions (true -> predicted

In [20]:
#chunk 7b
#checking on test run

import os  # file paths
import sys  # lets us add folders to Python's search path
import torch  # main library
import torch.nn as nn  # network building blocks

from google.colab import drive  # Colab Drive helper
drive.mount('/content/drive')  # connect Drive (safe to run twice)
PROJECT = "/content/drive/MyDrive/mfcad_project"  # project folder
sys.path.append(PROJECT)  # make helpers.py importable
import helpers  # our saving/progress helpers

NUM_CLASSES, HIDDEN, LAYERS, FHID, FLAYERS, GROUP = 25, 128, 6, 32, 2, 8  # MUST equal the training settings

def build_groups(fbs, mbs):  # merge batches into groups (same code as in training)
    groups = []  # result list
    for s in range(0, len(fbs), GROUP):  # step through batches in blocks
        foff, toff = 0, 0  # running offsets for faces and facets
        xs, ys, eis, ets, v2s, a2s, ffs = [], [], [], [], [], [], []  # collectors
        for fb, mb in zip(fbs[s:s + GROUP], mbs[s:s + GROUP]):  # each batch in this block
            assert fb["name"] == mb["name"], "face and mesh caches are out of step"  # same batch in both caches
            n, m = len(fb["y"]), len(mb["v2"])  # faces and facets
            xs.append(torch.as_tensor(fb["x"]).float()); ys.append(torch.as_tensor(fb["y"]).long())  # face inputs and classes
            eis.append(torch.as_tensor(fb["edge_index"]).long() + foff); ets.append(torch.as_tensor(fb["edge_type"]).long())  # face edges
            v2s.append(mb["v2"].float()); a2s.append(mb["a2"].long() + toff); ffs.append(mb["facet_face"].long() + foff)  # facet data
            foff += n; toff += m  # next batch starts after these
        x, y = torch.cat(xs), torch.cat(ys)  # faces of the group
        ei, et = torch.cat(eis, dim=1), torch.cat(ets)  # face edges of the group
        rel = []  # face edges split by type
        for t in range(3):  # convex, concave, smooth
            msk = et == t  # edges of this type
            src, dst = ei[0][msk], ei[1][msk]  # sender and receiver
            deg = torch.bincount(dst, minlength=foff).float().clamp(min=1)  # neighbours per face
            rel.append((src, dst, 1.0 / deg))  # store with 1/degree
        a2 = torch.cat(a2s, dim=1)  # facet edges of the group
        fdeg = torch.bincount(a2[1], minlength=toff).float().clamp(min=1)  # neighbours per facet
        ff = torch.cat(ffs)  # face of every facet
        cnt = torch.bincount(ff, minlength=foff).float().clamp(min=1)  # facets per face
        groups.append({"x": x, "y": y, "rel": rel, "v2": torch.cat(v2s), "frel": (a2[0], a2[1], 1.0 / fdeg), "ff": ff, "cnt": cnt})  # keep it
    return groups  # done

def load_groups(split):  # load face + mesh cache of a split and build groups
    fc = torch.load(f"{PROJECT}/data/cache_{split}.pt", weights_only=False)["batches"]  # face cache
    mc = torch.load(f"{PROJECT}/data/mesh_cache_{split}.pt", weights_only=False)["batches"]  # mesh cache
    return build_groups(fc, mc)  # merged groups

class FacetLayer(nn.Module):  # one round over neighbouring mesh facets
    def __init__(self, d):  # d = facet size
        super().__init__()  # standard setup
        self.self_lin = nn.Linear(d, d)  # own information
        self.nbr_lin = nn.Linear(d, d, bias=False)  # neighbour information
    def forward(self, h, rel):  # h = (facets, d)
        src, dst, inv = rel  # edge lists and 1/degree
        agg = torch.zeros_like(h).index_add_(0, dst, h[src]) * inv.unsqueeze(1)  # mean of neighbours
        return h + torch.relu(self.self_lin(h) + self.nbr_lin(agg))  # residual update

class Layer(nn.Module):  # one round over neighbouring faces
    def __init__(self, d):  # d = face size
        super().__init__()  # standard setup
        self.self_lin = nn.Linear(d, d)  # own information
        self.nbr_lin = nn.ModuleList([nn.Linear(d, d, bias=False) for _ in range(3)])  # one path per edge type
    def forward(self, h, rel):  # h = (faces, d)
        out = self.self_lin(h)  # start with the face itself
        for t, (src, dst, invdeg) in enumerate(rel):  # each edge type
            agg = torch.zeros_like(h).index_add_(0, dst, h[src]) * invdeg.unsqueeze(1)  # mean of neighbours
            out = out + self.nbr_lin[t](agg)  # add neighbour information
        return h + torch.relu(out)  # residual connection

class Net(nn.Module):  # facets -> faces -> classes
    def __init__(self):  # build the parts
        super().__init__()  # standard setup
        self.f_inp = nn.Linear(4, FHID)  # facet features -> facet size
        self.f_layers = nn.ModuleList([FacetLayer(FHID) for _ in range(FLAYERS)])  # facet rounds
        self.pool_lin = nn.Linear(FHID, HIDDEN)  # pooled facets -> face size
        self.inp = nn.Linear(16, HIDDEN)  # face features -> face size
        self.layers = nn.ModuleList([Layer(HIDDEN) for _ in range(LAYERS)])  # face rounds
        self.out = nn.Linear(HIDDEN, NUM_CLASSES)  # face size -> class scores
    def forward(self, g):  # g = one group
        f = torch.relu(self.f_inp(g["v2"]))  # embed facets
        for layer in self.f_layers:  # facet rounds
            f = layer(f, g["frel"])  # mix neighbouring facets
        pooled = torch.zeros(len(g["x"]), FHID).index_add_(0, g["ff"], f) / g["cnt"].unsqueeze(1)  # mean of each face's facets
        h = torch.relu(self.inp(g["x"]) + self.pool_lin(pooled))  # combine face and facet information
        for layer in self.layers:  # face rounds
            h = layer(h, g["rel"])  # mix neighbouring faces
        return self.out(h)  # class scores

model = Net()  # create the model
ck = torch.load(f"{PROJECT}/models/hier_ckpt.pt", weights_only=False)  # the final training checkpoint
model.load_state_dict(ck["model"])  # load the trained weights
print("checkpoint epoch:", ck["epoch"])  # should say 30
model.eval()  # evaluation mode

def evaluate(groups):  # accuracy, macro-F1, per-class IoU, confusion matrix
    cm = torch.zeros(NUM_CLASSES, NUM_CLASSES, dtype=torch.long)  # rows = true class
    with torch.no_grad():  # no gradients needed
        for g in groups:  # each group
            pred = model(g).argmax(1)  # predicted classes
            cm += torch.bincount(g["y"] * NUM_CLASSES + pred, minlength=NUM_CLASSES**2).reshape(NUM_CLASSES, NUM_CLASSES)  # add counts
    tp = cm.diag().float(); fp = cm.sum(0).float() - tp; fn = cm.sum(1).float() - tp  # per-class counts
    iou = tp / (tp + fp + fn).clamp(min=1); f1 = 2 * tp / (2 * tp + fp + fn).clamp(min=1)  # per-class IoU and F1
    return (tp.sum() / cm.sum()).item(), f1.mean().item(), iou, f1, cm  # results

acc, mf1, iou, _, _ = evaluate(load_groups("val"))  # sanity check on val first
print(f"SANITY val: accuracy {acc:.4f} | macro-F1 {mf1:.4f} | mean IoU {iou.mean():.4f}  (expect 0.9455 / 0.9229 / 0.8679)")  # compare with training output

acc, mf1, iou, f1, cm = evaluate(load_groups("test"))  # THE test evaluation (run once)
print(f"\nTEST accuracy {acc:.4f} | macro-F1 {mf1:.4f} | mean IoU {iou.mean():.4f} | faces {int(cm.sum())}")  # headline numbers
print("per-class IoU:", [round(v, 3) for v in iou.tolist()])  # IoU for classes 0-24
helpers.save_json({"test_accuracy": acc, "test_macro_f1": mf1, "test_mean_iou": iou.mean().item(),
                   "per_class_iou": iou.tolist(), "per_class_f1": f1.tolist(),
                   "confusion_matrix": cm.tolist(), "model": "hierarchical, hier_ckpt.pt epoch 30"}, "results", "chunk7b_hier_test.json")  # save to Drive
helpers.mark_done("chunk7b_test", "hierarchical model evaluated once on test")  # record progress

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
checkpoint epoch: 30
SANITY val: accuracy 0.9455 | macro-F1 0.9229 | mean IoU 0.8679  (expect 0.9455 / 0.9229 / 0.8679)

TEST accuracy 0.9454 | macro-F1 0.9222 | mean IoU 0.8668 | faces 259128
per-class IoU: [0.755, 0.981, 0.852, 0.845, 0.923, 0.745, 0.751, 0.915, 0.545, 0.899, 0.422, 0.958, 0.927, 0.914, 0.888, 0.943, 0.983, 0.888, 0.948, 0.947, 0.875, 0.972, 0.888, 0.914, 0.996]
Saved: /content/drive/MyDrive/mfcad_project/results/chunk7b_hier_test.json
Marked done: chunk7b_test


In [21]:
import sys  # lets us add folders to Python's search path
import torch  # main library
import torch.nn as nn  # network building blocks
import numpy as np  # array maths
from collections import Counter  # to find the most common mistake per part

from google.colab import drive  # Colab Drive helper
drive.mount('/content/drive')  # connect Drive (safe to run twice)
PROJECT = "/content/drive/MyDrive/mfcad_project"  # project folder
sys.path.append(PROJECT)  # make helpers.py importable
import helpers  # our saving/progress helpers

NUM_CLASSES, HIDDEN, LAYERS, FHID, FLAYERS = 25, 128, 6, 32, 2  # MUST equal the training settings
names = ["Chamfer", "Through hole", "Triangular passage", "Rectangular passage", "6-sided passage",
         "Triangular through slot", "Rectangular through slot", "Circular through slot",
         "Rectangular through step", "2-sided through step", "Slanted through step", "O-ring",
         "Blind hole", "Triangular pocket", "Rectangular pocket", "6-sided pocket",
         "Circular end pocket", "Rectangular blind slot", "Vertical circular end blind slot",
         "Horizontal circular end blind slot", "Triangular blind step", "Circular blind step",
         "Rectangular blind step", "Round", "Stock"]  # class names 0-24

def make_group(fb, mb):  # prepare ONE h5 batch as a graph group (same logic as in training, group size 1)
    x, y = torch.as_tensor(fb["x"]).float(), torch.as_tensor(fb["y"]).long()  # face inputs and classes
    ei, et = torch.as_tensor(fb["edge_index"]).long(), torch.as_tensor(fb["edge_type"]).long()  # face edges and types
    n = len(y)  # number of faces
    rel = []  # face edges split by type
    for t in range(3):  # convex, concave, smooth
        msk = et == t  # edges of this type
        src, dst = ei[0][msk], ei[1][msk]  # sender and receiver
        deg = torch.bincount(dst, minlength=n).float().clamp(min=1)  # neighbours per face
        rel.append((src, dst, 1.0 / deg))  # store with 1/degree
    a2 = mb["a2"].long()  # facet edges
    m = len(mb["v2"])  # number of facets
    fdeg = torch.bincount(a2[1], minlength=m).float().clamp(min=1)  # neighbours per facet
    ff = mb["facet_face"].long()  # face of every facet
    cnt = torch.bincount(ff, minlength=n).float().clamp(min=1)  # facets per face
    return {"x": x, "y": y, "rel": rel, "v2": mb["v2"].float(), "frel": (a2[0], a2[1], 1.0 / fdeg), "ff": ff, "cnt": cnt}  # one group

class FacetLayer(nn.Module):  # one round over neighbouring mesh facets
    def __init__(self, d):  # d = facet size
        super().__init__()  # standard setup
        self.self_lin = nn.Linear(d, d)  # own information
        self.nbr_lin = nn.Linear(d, d, bias=False)  # neighbour information
    def forward(self, h, rel):  # h = (facets, d)
        src, dst, inv = rel  # edge lists and 1/degree
        agg = torch.zeros_like(h).index_add_(0, dst, h[src]) * inv.unsqueeze(1)  # mean of neighbours
        return h + torch.relu(self.self_lin(h) + self.nbr_lin(agg))  # residual update

class Layer(nn.Module):  # one round over neighbouring faces
    def __init__(self, d):  # d = face size
        super().__init__()  # standard setup
        self.self_lin = nn.Linear(d, d)  # own information
        self.nbr_lin = nn.ModuleList([nn.Linear(d, d, bias=False) for _ in range(3)])  # one path per edge type
    def forward(self, h, rel):  # h = (faces, d)
        out = self.self_lin(h)  # start with the face itself
        for t, (src, dst, invdeg) in enumerate(rel):  # each edge type
            agg = torch.zeros_like(h).index_add_(0, dst, h[src]) * invdeg.unsqueeze(1)  # mean of neighbours
            out = out + self.nbr_lin[t](agg)  # add neighbour information
        return h + torch.relu(out)  # residual connection

class Net(nn.Module):  # facets -> faces -> classes
    def __init__(self):  # build the parts
        super().__init__()  # standard setup
        self.f_inp = nn.Linear(4, FHID)  # facet features -> facet size
        self.f_layers = nn.ModuleList([FacetLayer(FHID) for _ in range(FLAYERS)])  # facet rounds
        self.pool_lin = nn.Linear(FHID, HIDDEN)  # pooled facets -> face size
        self.inp = nn.Linear(16, HIDDEN)  # face features -> face size
        self.layers = nn.ModuleList([Layer(HIDDEN) for _ in range(LAYERS)])  # face rounds
        self.out = nn.Linear(HIDDEN, NUM_CLASSES)  # face size -> class scores
    def forward(self, g):  # g = one group
        f = torch.relu(self.f_inp(g["v2"]))  # embed facets
        for layer in self.f_layers:  # facet rounds
            f = layer(f, g["frel"])  # mix neighbouring facets
        pooled = torch.zeros(len(g["x"]), FHID).index_add_(0, g["ff"], f) / g["cnt"].unsqueeze(1)  # mean of each face's facets
        h = torch.relu(self.inp(g["x"]) + self.pool_lin(pooled))  # combine face and facet information
        for layer in self.layers:  # face rounds
            h = layer(h, g["rel"])  # mix neighbouring faces
        return self.out(h)  # class scores

model = Net()  # create the model
model.load_state_dict(torch.load(f"{PROJECT}/models/hier_ckpt.pt", weights_only=False)["model"])  # trained weights
model.eval()  # evaluation mode

fc = torch.load(f"{PROJECT}/data/cache_test.pt", weights_only=False)["batches"]  # test face cache
mc = torch.load(f"{PROJECT}/data/mesh_cache_test.pt", weights_only=False)["batches"]  # test mesh cache

parts, correct_total, faces_total = [], 0, 0  # one record per part, plus overall counts
with torch.no_grad():  # no gradients needed
    for fb, mb in zip(fc, mc):  # one h5 batch at a time
        assert fb["name"] == mb["name"], "caches out of step"  # same batch in both caches
        g = make_group(fb, mb)  # prepare the batch
        pred = model(g).argmax(1).numpy()  # predicted class per face
        true = g["y"].numpy()  # true class per face
        fm = fb["face_model"].numpy().astype(int)  # part number per face (from idx)
        nparts = len(fb["model_ids"])  # parts in this batch
        wrong = (pred != true).astype(float)  # 1 where the face is wrong
        n_faces = np.bincount(fm, minlength=nparts)  # faces per part
        n_wrong = np.bincount(fm, weights=wrong, minlength=nparts).astype(int)  # wrong faces per part
        correct_total += int((pred == true).sum()); faces_total += len(true)  # overall counts
        for p in range(nparts):  # one record per part
            sel = fm == p  # faces of this part
            mist = Counter(zip(true[sel][wrong[sel] == 1].tolist(), pred[sel][wrong[sel] == 1].tolist()))  # (true, predicted) pairs of its mistakes
            top = mist.most_common(1)[0] if mist else None  # most common mistake in this part
            parts.append({"model_id": str(fb["model_ids"][p]), "batch": str(fb["name"]), "faces": int(n_faces[p]),
                          "wrong": int(n_wrong[p]),
                          "top_mistake": None if top is None else [names[top[0][0]], names[top[0][1]], top[1]]})  # keep it

print(f"SANITY: accuracy {correct_total / faces_total:.4f} on {faces_total} faces (test run gave 0.9454)")  # must match 7b
w = np.array([q["wrong"] for q in parts]); f_ = np.array([q["faces"] for q in parts])  # arrays for statistics
print("parts:", len(parts), "| parts with 0 wrong faces:", int((w == 0).sum()), f"({(w == 0).mean():.1%})")  # fully correct parts
print("parts with 1-2 wrong faces:", int(((w >= 1) & (w <= 2)).sum()), "| 3-5:", int(((w >= 3) & (w <= 5)).sum()), "| 6 or more:", int((w >= 6).sum()))  # error spread
order = np.argsort(-w)  # parts sorted from most wrong to least
print(f"Worst 10% of parts ({len(parts)//10}) hold {w[order[:len(parts)//10]].sum() / w.sum():.1%} of all wrong faces")  # concentration of errors
print("\n15 worst parts (model id | batch | wrong/faces | most common mistake: true -> predicted x count):")  # section title
for i in order[:15]:  # the 15 worst parts
    q = parts[i]  # record
    print(f"  {q['model_id']} | batch {q['batch']} | {q['wrong']}/{q['faces']} | {q['top_mistake'][0]} -> {q['top_mistake'][1]} x{q['top_mistake'][2]}")  # one line per part

helpers.save_json(parts, "results", "chunk7c_test_parts.json")  # save all per-part results to Drive
helpers.mark_done("chunk7c_parts", "per-part error analysis of hierarchical model on test")  # record progress

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
SANITY: accuracy 0.9454 on 259128 faces (test run gave 0.9454)
parts: 8632 | parts with 0 wrong faces: 4375 (50.7%)
parts with 1-2 wrong faces: 2141 | 3-5: 1449 | 6 or more: 667
Worst 10% of parts (863) hold 44.9% of all wrong faces

15 worst parts (model id | batch | wrong/faces | most common mistake: true -> predicted x count):
  51713 | batch 206 | 23/29 | Blind hole -> Circular end pocket x4
  143 | batch 103 | 22/38 | Triangular passage -> 6-sided passage x8
  22569 | batch 152 | 21/65 | O-ring -> Blind hole x13
  24979 | batch 226 | 20/61 | 6-sided passage -> Rectangular passage x10
  46382 | batch 102 | 20/63 | 6-sided passage -> Rectangular passage x15
  4049 | batch 66 | 19/49 | 6-sided pocket -> Rectangular pocket x12
  26986 | batch 281 | 19/64 | 6-sided pocket -> Rectangular pocket x13
  7470 | batch 294 | 18/61 | 6-sided passage -> Triangular pas